<a href="https://colab.research.google.com/github/alxmzr/Colab/blob/main/%D0%A1ycle_Mirror.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- EXPANDED SETTINGS FOR LARGE-SCALE INTRADAY OPTIMIZATION ---
TICKER = "AUDUSD=X"
# Expanded range to 1-1440 hours (up to 60 days)
LOOKBACK_RANGE = range(1, 24)
FORECAST_HOURS = 12

# 1. Download Hourly Data
print(f"Downloading hourly data for {TICKER}...")
# 730 days is the maximum allowed for 1h interval via yfinance
df_hourly = yf.download(TICKER, period="730d", interval="1h", auto_adjust=True)
if isinstance(df_hourly.columns, pd.MultiIndex): df_hourly.columns = df_hourly.columns.get_level_values(0)
df_hourly = df_hourly['Close'].to_frame().dropna()

# 2. Grid Search for Optimal Hourly Lookback
hourly_results = []

print(f"Searching for optimal intraday cycle window (1-24h)... ")

for lb in LOOKBACK_RANGE:
    current_pattern = df_hourly['Close'].iloc[-lb:].values
    if len(current_pattern) < lb or np.std(current_pattern) == 0: continue

    # Define search space excluding the forecast and current window
    search_space = df_hourly.iloc[:-FORECAST_HOURS - lb]
    best_corr = -1
    best_idx = -1

    # For large ranges, we skip to find best match efficiently
    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        if np.std(sample) > 0:
            corr = np.corrcoef(current_pattern, sample)[0, 1]
            if corr > best_corr:
                best_corr = corr
                best_idx = i

    hourly_results.append({"Lookback_Hours": lb, "Correlation": best_corr, "Index": best_idx})

res_df = pd.DataFrame(hourly_results)
best_lb_row = res_df.loc[res_df['Correlation'].idxmax()]
opt_lb = int(best_lb_row['Lookback_Hours'])
opt_corr = best_lb_row['Correlation']
opt_idx = int(best_lb_row['Index'])

print(f"\nOptimal Window: {opt_lb} hours")
print(f"Max Correlation: {opt_corr:.2%}")

# 3. Visualization
best_sample = df_hourly.iloc[opt_idx : opt_idx + opt_lb + FORECAST_HOURS]
start_time = best_sample.index[0].strftime('%Y-%m-%d %H:%M')

target_final = df_hourly['Close'].iloc[-opt_lb:].values
ratio = target_final[-1] / best_sample['Close'].values[opt_lb-1]
projected_vals = best_sample['Close'].values * ratio

fig = go.Figure()
x_axis = np.arange(-opt_lb + 1, FORECAST_HOURS + 1)

fig.add_trace(go.Scatter(x=x_axis[:opt_lb], y=target_final, name="Current (Hourly)", line=dict(color='black', width=4)))
fig.add_trace(go.Scatter(x=x_axis, y=projected_vals, name=f"Fractal (Start: {start_time})", line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST_HOURS, fillcolor="rgba(255,0,0,0.05)", annotation_text="12H FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Global Hourly Fractal Optimization (1-24h Search): {TICKER}</b><br><sup>Best Window: {opt_lb} Hours | Correlation: {opt_corr:.2%}</sup>",
    xaxis_title="Hours from Now", yaxis_title="Price (USD)", template="plotly_white"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal intraday cycle window (1-24h)... 

Optimal Window: 2 hours
Max Correlation: 100.00%


In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- АДАВТИРАВАННЫЕ НАСТРАЙКИ ---
TICKER = "ETH-USD"
LOOKBACK = 60       # Дни для анализа
FORECAST = 45       # Прогноз
TOP_N = 3           # Количество лучших циклов

# Загрузка данных
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

current_df = df.iloc[-LOOKBACK:]
current_prices = current_df['Close'].values
current_dates = current_df.index

# Нормализация (Z-score) для поиска
target_norm = (current_prices - np.mean(current_prices)) / np.std(current_prices)

results = []
search_space = df.iloc[:-FORECAST]
for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
    sample_norm = (sample - np.mean(sample)) / np.std(sample)
    score = np.mean((target_norm - sample_norm)**2)
    results.append((score, i))

results.sort(key=lambda x: x[0])
best_cycles_data = []
used_indices = []
for score, idx in results:
    if not any(abs(idx - used_idx) < 30 for used_idx in used_indices):
        # Расчет корреляции Пирсона для этого участка
        hist_prices = search_space.iloc[idx : idx + LOOKBACK]['Close'].values
        correlation = np.corrcoef(current_prices, hist_prices)[0, 1]
        best_cycles_data.append((idx, correlation))
        used_indices.append(idx)
    if len(best_cycles_data) == TOP_N: break

# Подготовка данных для интервалов
forecast_matrix = []
future_dates = [current_dates[-1] + timedelta(days=i) for i in range(1, FORECAST + 1)]

# Визуализация
fig = go.Figure()
fig.add_trace(go.Scatter(x=current_dates, y=current_prices, name="ТЕКУЩАЯ ЦЕНА", line=dict(color='black', width=4)))

colors = ['red', 'blue', 'green']
for i, (idx, corr) in enumerate(best_cycles_data):
    raw_values = search_space.iloc[idx : idx + LOOKBACK + FORECAST]['Close'].values
    ratio = current_prices[-1] / raw_values[LOOKBACK-1]
    proj_values = raw_values[LOOKBACK:] * ratio
    forecast_matrix.append(proj_values)

    start_date = search_space.index[idx].strftime('%Y-%m-%d')
    fig.add_trace(go.Scatter(
        x=future_dates,
        y=proj_values,
        name=f"Цикл {i+1} ({start_date}) Corr: {corr:.2%}",
        line=dict(color=colors[i], width=2, dash='dot')
    ))

forecast_matrix = np.array(forecast_matrix)
mean_forecast = np.mean(forecast_matrix, axis=0)
upper_bound = np.max(forecast_matrix, axis=0)
lower_bound = np.min(forecast_matrix, axis=0)

# Доверительный интервал
fig.add_trace(go.Scatter(
    x=future_dates + future_dates[::-1],
    y=list(upper_bound) + list(lower_bound)[::-1],
    fill='toself', fillcolor='rgba(0,100,80,0.1)', line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip", name="Диапазон (Top-3)"
))

# Средний прогноз
fig.add_trace(go.Scatter(x=future_dates, y=mean_forecast, name="СРЕДНИЙ ПРОГНОЗ", line=dict(color='orange', width=3)))

fig.update_layout(
    title=f"<b>Прогноз {TICKER} с мат. сходством (Pearson Correlation)</b>",
    xaxis_title="Дата", yaxis_title="Цена",
    template="plotly_white", hovermode="x unified"
)
if "USD" in TICKER: fig.update_yaxes(type="log")
fig.show()

[*********************100%***********************]  1 of 1 completed


In [4]:
# --- VISUALIZING MATCH QUALITY (ACTUAL VS MATCHES) ---
import plotly.graph_objects as go
import numpy as np

fig_match = go.Figure()

# 1. Current Actual Prices (Normalized for shape comparison)
current_prices_norm = (current_prices - np.mean(current_prices)) / np.std(current_prices)
fig_match.add_trace(go.Scatter(
    x=np.arange(LOOKBACK),
    y=current_prices_norm,
    name="Actual Price (Z-Score)",
    line=dict(color='black', width=4)
))

# 2. Top 3 Best Matching Historical Fractals (Normalized)
colors = ['red', 'blue', 'green']
for i, (idx, corr) in enumerate(best_cycles_data):
    hist_sample = df.iloc[idx : idx + LOOKBACK]['Close'].values
    hist_norm = (hist_sample - np.mean(hist_sample)) / np.std(hist_sample)
    start_date = df.index[idx].strftime('%Y-%m-%d')

    fig_match.add_trace(go.Scatter(
        x=np.arange(LOOKBACK),
        y=hist_norm,
        name=f"Match #{i+1} (Start: {start_date})",
        line=dict(color=colors[i], width=2, dash='dot'),
        opacity=0.7
    ))

fig_match.update_layout(
    title=f"<b>Fractal Fit Analysis: Actual vs. Historical Matches ({TICKER})</b><br><sup>Comparing normalized shapes over the {LOOKBACK}-day lookback period</sup>",
    xaxis_title="Days in Lookback Window",
    yaxis_title="Normalized Price (Z-Score)",
    template="plotly_white",
    hovermode="x unified"
)

fig_match.show()

In [5]:
# --- VISUALIZING MATCH QUALITY (ACTUAL VS MATCHES) ---
import plotly.graph_objects as go

fig_match = go.Figure()

# 1. Current Actual Prices (Normalized for shape comparison)
current_prices_norm = (current_prices - np.mean(current_prices)) / np.std(current_prices)
fig_match.add_trace(go.Scatter(
    x=np.arange(LOOKBACK),
    y=current_prices_norm,
    name="Actual Price (Z-Score)",
    line=dict(color='black', width=4)
))

# 2. Top 3 Best Matching Historical Fractals (Normalized)
colors = ['red', 'blue', 'green']
for i, (idx, corr) in enumerate(best_cycles_data):
    hist_sample = df.iloc[idx : idx + LOOKBACK]['Close'].values
    hist_norm = (hist_sample - np.mean(hist_sample)) / np.std(hist_sample)
    start_date = df.index[idx].strftime('%Y-%m-%d')

    fig_match.add_trace(go.Scatter(
        x=np.arange(LOOKBACK),
        y=hist_norm,
        name=f"Match #{i+1} (Start: {start_date})",
        line=dict(color=colors[i], width=2, dash='dot'),
        opacity=0.7
    ))

fig_match.update_layout(
    title=f"<b>Fractal Fit Analysis: Actual vs. Historical Matches ({TICKER})</b><br><sup>Comparing normalized shapes over the {LOOKBACK}-day lookback period</sup>",
    xaxis_title="Days in Lookback Window",
    yaxis_title="Normalized Price (Z-Score)",
    template="plotly_white",
    hovermode="x unified"
)

fig_match.show()

### Forecast Summary Table
This table presents the projected price levels for key dates in the future based on the Top-3 matching fractals.

In [6]:
# --- FORECAST TABLE GENERATION ---

# Create a DataFrame for the forecast
forecast_df = pd.DataFrame({
    'Date': [d.strftime('%Y-%m-%d') for d in future_dates],
    'Mean_Projected': mean_forecast,
    'Low_Scenario': lower_bound,
    'High_Scenario': upper_bound
})

# Filter for weekly milestones (every 7 days) and the final day
steps = list(range(0, len(forecast_df), 7))
if (len(forecast_df) - 1) not in steps:
    steps.append(len(forecast_df) - 1)

summary_table = forecast_df.iloc[steps].copy()

# Round for readability
summary_table = summary_table.round(2)

print(f"Forecast Summary for {TICKER} (Next {FORECAST} Days):")
display(summary_table)

Forecast Summary for ETH-USD (Next 45 Days):


,Date,Mean_Projected,Low_Scenario,High_Scenario
0,2026-05-12,2480.48,2410.97,2611.86
7,2026-05-19,2474.11,2347.99,2692.37
14,2026-05-26,2572.36,2413.43,2841.32
21,2026-06-02,2514.72,2096.30,2818.60
28,2026-06-09,2419.91,1943.25,2794.24
35,2026-06-16,2470.16,2020.65,2951.15
42,2026-06-23,2384.08,2005.10,2691.00
44,2026-06-25,2398.30,1928.02,2636.09


In [7]:
# --- VISUALIZING MATCH QUALITY (ACTUAL VS MATCHES) ---
import plotly.graph_objects as go
import numpy as np

fig_match = go.Figure()

# 1. Current Actual Prices (Normalized for shape comparison)
current_prices_norm = (current_prices - np.mean(current_prices)) / np.std(current_prices)
fig_match.add_trace(go.Scatter(
    x=np.arange(LOOKBACK),
    y=current_prices_norm,
    name="Actual Price (Z-Score)",
    line=dict(color='black', width=4)
))

# 2. Top 3 Best Matching Historical Fractals (Normalized)
colors = ['red', 'blue', 'green']
# Fixed: using best_cycles_data and unpacking (idx, corr)
for i, (idx, corr) in enumerate(best_cycles_data):
    hist_sample = df.iloc[idx : idx + LOOKBACK]['Close'].values
    hist_norm = (hist_sample - np.mean(hist_sample)) / np.std(hist_sample)
    start_date = df.index[idx].strftime('%Y-%m-%d')

    fig_match.add_trace(go.Scatter(
        x=np.arange(LOOKBACK),
        y=hist_norm,
        name=f"Match #{i+1} (Start: {start_date})",
        line=dict(color=colors[i], width=2, dash='dot'),
        opacity=0.7
    ))

fig_match.update_layout(
    title=f"<b>Fractal Fit Analysis: Actual vs. Historical Matches ({TICKER})</b><br><sup>Comparing normalized shapes over the {LOOKBACK}-day lookback period</sup>",
    xaxis_title="Days in Lookback Window",
    yaxis_title="Normalized Price (Z-Score)",
    template="plotly_white",
    hovermode="x unified"
)

fig_match.show()

In [8]:
# --- CONSOLIDATED COMPARISON ANALYSIS ---

# 1. Recalculate Backtest Top-3 (100 days ago)
BACKTEST_DAYS_AGO = 100
cutoff_idx = len(df) - BACKTEST_DAYS_AGO
test_data = df.iloc[:cutoff_idx]

target_series_bt = test_data['Close'].iloc[-LOOKBACK:].values
target_norm_bt = (target_series_bt - np.mean(target_series_bt)) / np.std(target_series_bt)

search_space_bt = test_data.iloc[:-FORECAST]
results_bt = []
for i in range(len(search_space_bt) - LOOKBACK):
    sample = search_space_bt.iloc[i : i + LOOKBACK]['Close'].values
    sample_norm = (sample - np.mean(sample)) / np.std(sample)
    score = np.mean((target_norm_bt - sample_norm)**2)
    results_bt.append((score, i))

results_bt.sort(key=lambda x: x[0])
backtest_best_indices = []
used_bt = []
for s, idx in results_bt:
    if not any(abs(idx - u) < 30 for u in used_bt):
        backtest_best_indices.append(idx)
        used_bt.append(idx)
    if len(backtest_best_indices) == TOP_N: break

# 2. Extract Current Model Data
current_best_indices = [item[0] for item in best_cycles_data]
current_best_dates = [df.index[idx].strftime('%Y-%m-%d') for idx in current_best_indices]
backtest_best_dates = [df.index[idx].strftime('%Y-%m-%d') for idx in backtest_best_indices]

# 3. Build Comparison Table
comparison_df = pd.DataFrame({
    'Rank': [f'#{i+1}' for i in range(len(current_best_dates))],
    'Current Model Cycle Start': current_best_dates,
    'Backtest (100d ago) Cycle Start': backtest_best_dates
})

print("Comparison of Top-3 Fractal Matches:")
display(comparison_df)

# Check for overlap
overlap = set(current_best_dates).intersection(set(backtest_best_dates))
if overlap:
    print(f"\nCommon cycles found in both models: {list(overlap)}")
else:
    print("\nNo exact overlapping cycles found. This is normal as the current model has 100 more days of data available.")

Comparison of Top-3 Fractal Matches:


,Rank,Current Model Cycle Start,Backtest (100d ago) Cycle Start
0,#1,2021-07-03,2022-09-21
1,#2,2019-04-15,2019-10-01
2,#3,2019-01-15,2018-11-22



No exact overlapping cycles found. This is normal as the current model has 100 more days of data available.


In [9]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- НАСТРОЙКИ БЭКТЕСТА ---
BACKTEST_DAYS_AGO = 100  # Точка в прошлом для проверки
LOOKBACK = 60
FORECAST = 60
TOP_N = 3

# Используем уже загруженные данные df
# Точка отсечки для теста
cutoff_idx = len(df) - BACKTEST_DAYS_AGO
test_data = df.iloc[:cutoff_idx]
actual_after = df.iloc[cutoff_idx : cutoff_idx + FORECAST]

# Текущий паттерн на момент отсечки
target_series = test_data['Close'].iloc[-LOOKBACK:].values
target_norm = (target_series - np.mean(target_series)) / np.std(target_series)
target_dates = test_data.index[-LOOKBACK:]

# Поиск по истории ДО точки отсечки
search_space = test_data.iloc[:-FORECAST]
results = []
for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
    sample_norm = (sample - np.mean(sample)) / np.std(sample)
    score = np.mean((target_norm - sample_norm)**2)
    results.append((score, i))

results.sort(key=lambda x: x[0])
best_indices = []
used = []
for s, idx in results:
    if not any(abs(idx - u) < 30 for u in used):
        best_indices.append(idx)
        used.append(idx)
    if len(best_indices) == TOP_N: break

# Сбор прогнозов
forecast_matrix = []
future_dates = [target_dates[-1] + timedelta(days=i) for i in range(1, FORECAST + 1)]

for idx in best_indices:
    raw = search_space.iloc[idx : idx + LOOKBACK + FORECAST]['Close'].values
    ratio = target_series[-1] / raw[LOOKBACK-1]
    forecast_matrix.append(raw[LOOKBACK:] * ratio)

forecast_matrix = np.array(forecast_matrix)
mean_f = np.mean(forecast_matrix, axis=0)

# --- ВИЗУАЛИЗАЦИЯ БЭКТЕСТА ---
fig = go.Figure()

# Реальные данные (до и после)
fig.add_trace(go.Scatter(x=target_dates, y=target_series, name="Факт (до теста)", line=dict(color='black', width=3)))
fig.add_trace(go.Scatter(x=actual_after.index, y=actual_after['Close'], name="Факт (реальное будущее)", line=dict(color='orange', width=3)))

# Прогноз
fig.add_trace(go.Scatter(x=future_dates, y=mean_f, name="Средний прогноз фракталов", line=dict(color='green', dash='dash')))

fig.update_layout(
    title="<b>Бэктест: Сравнение прогноза с реальными данными в прошлом</b>",
    xaxis_title="Дата", yaxis_title="Цена",
    template="plotly_white", hovermode="x unified"
)
fig.show()

In [10]:
import itertools

# --- ПАРАМЕТРЫ ДЛЯ ТЕСТИРОВАНИЯ ---
lookback_options = [30, 60, 90]
forecast_options = [30, 45, 60]
BACKTEST_DAYS = 100

results_grid = []

# Отрезаем данные для теста
cutoff_idx = len(df) - BACKTEST_DAYS
test_data = df.iloc[:cutoff_idx]
actual_future = df.iloc[cutoff_idx : cutoff_idx + max(forecast_options)]

print("Запуск оптимизации сетки...")

for lb, fc in itertools.product(lookback_options, forecast_options):
    # Текущий паттерн
    target = test_data['Close'].iloc[-lb:].values
    target_norm = (target - np.mean(target)) / np.std(target)

    # Поиск лучшего фрактала в истории
    best_score = float('inf')
    best_idx = -1
    search_space = test_data.iloc[:-fc]

    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        sample_norm = (sample - np.mean(sample)) / np.std(sample)
        score = np.mean((target_norm - sample_norm)**2)
        if score < best_score:
            best_score = score
            best_idx = i

    # Проверка точности прогноза
    if best_idx != -1:
        raw_hist = search_space.iloc[best_idx : best_idx + lb + fc]['Close'].values
        ratio = target[-1] / raw_hist[lb-1]
        pred = raw_hist[lb:] * ratio

        # Считаем ошибку относительно реальности
        actual = df['Close'].iloc[cutoff_idx : cutoff_idx + fc].values
        rmse = np.sqrt(np.mean((pred - actual)**2))
        results_grid.append({'Lookback': lb, 'Forecast': fc, 'RMSE': rmse})

# Вывод результатов
optimization_df = pd.DataFrame(results_grid).sort_values('RMSE')
print("\nЛучшие параметры по итогам бэктеста:")
display(optimization_df.head())

# Визуализация матрицы ошибок
pivot_df = optimization_df.pivot(index='Lookback', columns='Forecast', values='RMSE')
fig = go.Figure(data=go.Heatmap(
    z=pivot_df.values,
    x=[f'Forecast {c}' for c in pivot_df.columns],
    y=[f'Lookback {r}' for r in pivot_df.index],
    colorscale='Viridis'))
fig.update_layout(title='Карта ошибок (RMSE) параметров фрактала')
fig.show()

Запуск оптимизации сетки...

Лучшие параметры по итогам бэктеста:


,Lookback,Forecast,RMSE
6,90,30,203.931475
7,90,45,431.003691
8,90,60,460.208130
0,30,30,678.834893
4,60,45,699.264495


In [11]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- НАСТРОЙКИ ---
TICKER = "BTC-USD"          # Можно заменить на ^GSPC (S&P500)
LOOKBACK_WINDOW = 360       # Длина отрезка для сравнения (например, последние 60 дней)
FORECAST_WINDOW = 30       # На сколько дней вперед мы хотим видеть прогноз из прошлого

# --- 1. ЗАГРУЗКА ДАННЫХ ---
df = yf.download(TICKER, start="2010-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Берем текущее движение (последние 60 дней)
current_move = df['Close'].iloc[-LOOKBACK_WINDOW:].values
current_move_norm = (current_move / current_move[0])

# --- 2. ПОИСК ПОХОЖЕГО ЦИКЛА (АЛГОРИТМ) ---
best_score = float('inf')
best_idx = -1

# Проходим по всей истории и ищем минимальную разницу (MSE)
for i in range(len(df) - LOOKBACK_WINDOW - FORECAST_WINDOW):
    past_move = df['Close'].iloc[i : i + LOOKBACK_WINDOW].values
    past_move_norm = (past_move / past_move[0])

    # Считаем разницу между текущим фракталом и историческим
    score = np.sum((current_move_norm - past_move_norm)**2)

    if score < best_score:
        best_score = score
        best_idx = i

# Извлекаем найденный цикл + его будущее продолжение
found_cycle = df.iloc[best_idx : best_idx + LOOKBACK_WINDOW + FORECAST_WINDOW]
found_date = found_cycle.index[0].strftime('%Y-%m-%d')
found_values_norm = (found_cycle['Close'].values / found_cycle['Close'].values[0]) * 100

# Текущие данные для графика (в % от старта окна)
current_plot = (current_move / current_move[0]) * 100

# --- 3. ВИЗУАЛИЗАЦИЯ ---
fig = go.Figure()

# Линия текущего движения
fig.add_trace(go.Scatter(
    x=np.arange(LOOKBACK_WINDOW),
    y=current_plot,
    name="ТЕКУЩЕЕ ДВИЖЕНИЕ",
    line=dict(color='black', width=4)
))

# Линия найденного исторического цикла
fig.add_trace(go.Scatter(
    x=np.arange(LOOKBACK_WINDOW + FORECAST_WINDOW),
    y=found_values_norm,
    name=f"НАЙДЕННЫЙ ЦИКЛ (от {found_date})",
    line=dict(color='red', width=2, dash='dot')
))

# Выделяем зону прогноза
fig.add_vrect(
    x0=LOOKBACK_WINDOW-1, x1=LOOKBACK_WINDOW + FORECAST_WINDOW - 1,
    fillcolor="green", opacity=0.1, layer="below", line_width=0,
    annotation_text="ВЕРОЯТНОЕ БУДУЩЕЕ"
)

fig.update_layout(
    title=f"Поиск фрактала для {TICKER} (Найден цикл от {found_date})",
    xaxis_title="Дни",
    yaxis_title="Изменение цены (%)",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()


[*********************100%***********************]  1 of 1 completed


In [12]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta

# --- 1. НАСТРОЙКИ ---
TICKER = "BTC-USD"           # Инструмент (S&P 500)
LOOKBACK = 60              # Окно сравнения (насколько длинный текущий кусок берем, дни)
FORECAST = 40              # Прогноз (на сколько дней вперед продлеваем историю)

# --- 2. ЗАГРУЗКА ДАННЫХ ---
df = yf.download(TICKER, start="1970-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

def find_best_fractal(target_series, search_space, label):
    """Ищет один самый похожий участок в заданном пространстве данных"""
    best_score = float('inf')
    best_idx = -1

    target_norm = target_series / target_series[0]

    # Скользящее окно по истории
    for i in range(len(search_space) - LOOKBACK - FORECAST):
        sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
        sample_norm = sample / sample[0]

        # Считаем среднеквадратичную ошибку (MSE)
        score = np.mean((target_norm - sample_norm)**2)

        if score < best_score:
            best_score = score
            best_idx = i

    if best_idx != -1:
        res = search_space.iloc[best_idx : best_idx + LOOKBACK + FORECAST]['Close'].values
        date_str = search_space.index[best_idx].strftime('%Y-%m-%d')
        return (res / res[0]) * 100, date_str
    return None, None

# --- 3. ПОДГОТОВКА ПОИСКОВЫХ ЗОН ---
current_data = df['Close'].iloc[-LOOKBACK:]
four_years_ago = df.index[-1] - timedelta(days=4*365)
one_year_ago = df.index[-1] - timedelta(days=365)

# Зоны поиска
zones = {
    "Максимально возможный (вся история)": df,
    "За последние 4 года": df[df.index >= four_years_ago],
    "Годовой цикл": df[df.index >= one_year_ago]
}

# --- 4. ВИЗУАЛИЗАЦИЯ ---
fig = go.Figure()

# Текущая цена (черная жирная линия)
current_plot = (current_data.values / current_data.values[0]) * 100
fig.add_trace(go.Scatter(x=np.arange(LOOKBACK), y=current_plot,
                         name="ТЕКУЩЕЕ ДВИЖЕНИЕ", line=dict(color='black', width=5)))

colors = ['red', 'blue', 'green']
for i, (name, space) in enumerate(zones.items()):
    # Исключаем последние дни, чтобы не найти «самого себя»
    search_data = space.iloc[:-FORECAST]

    values, date_found = find_best_fractal(current_data.values, search_data, name)

    if values is not None:
        fig.add_trace(go.Scatter(
            x=np.arange(LOOKBACK + FORECAST),
            y=values,
            name=f"{name} (от {date_found})",
            line=dict(color=colors[i], width=2, dash='dot'),
            opacity=0.8
        ))

# Оформление
fig.update_layout(
    title=f"<b>Мульти-цикловой анализ {TICKER}</b><br><sup>Поиск фракталов в разных временных масштабах</sup>",
    xaxis_title="Торговые дни (от точки отсчета)",
    yaxis_title="Относительное изменение (%)",
    template="plotly_white",
    hovermode="x unified",
    shapes=[dict(type="line", x0=LOOKBACK-1, x1=LOOKBACK-1, y0=min(current_plot)*0.95, y1=max(current_plot)*1.05,
                 line=dict(color="Gray", dash="dash"))]
)

fig.show()

[*********************100%***********************]  1 of 1 completed


In [13]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta

# --- 1. НАСТРОЙКИ ПОД КРИПТО ---
# Выберите нужный тикер: "BTC-USD" или "ETH-USD"
TICKER = "BTC-USD"
LOOKBACK = 45   # Окно анализа (1.5 месяца)
FORECAST = 30   # Прогноз на месяц вперед

# Загружаем данные (крипта торгуется ежедневно, поэтому данных больше)
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

def find_crypto_fractal(target_series, search_space):
    best_score = float('inf')
    best_idx = -1

    # Используем логарифмическое изменение для крипты (лучше передает характер движения)
    target_log = np.log(target_series)
    target_norm = (target_log - np.mean(target_log)) / np.std(target_log)

    for i in range(len(search_space) - LOOKBACK - FORECAST):
        sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
        sample_log = np.log(sample)
        sample_norm = (sample_log - np.mean(sample_log)) / np.std(sample_log)

        score = np.mean((target_norm - sample_norm)**2)
        if score < best_score:
            best_score = score
            best_idx = i

    if best_idx != -1:
        raw_values = search_space.iloc[best_idx : best_idx + LOOKBACK + FORECAST]['Close'].values
        # Совмещаем по последней цене (Price Action стыковка)
        ratio = target_series[-1] / raw_values[LOOKBACK-1]
        adjusted_values = raw_values * ratio

        corr = np.corrcoef(target_series, raw_values[:LOOKBACK])[0, 1]
        date_str = search_space.index[best_idx].strftime('%Y-%m-%d')
        return adjusted_values, date_str, corr
    return None, None, None

# --- 2. ЗАПУСК ПОИСКА ---
current_data = df['Close'].iloc[-LOOKBACK:].values
fig = go.Figure()

# Основной график BTC/ETH
fig.add_trace(go.Scatter(y=current_data, name=f"ТЕКУЩИЙ {TICKER}",
                         line=dict(color='#F7931A' if "BTC" in TICKER else '#627EEA', width=4)))

# Поиск зон
search_zones = [
    ("Глобальный цикл", df, 'red'),
    ("Цикл после Халвинга (последние 4г)", df[df.index >= df.index[-1] - timedelta(days=4*365)], 'blue'),
    ("Краткосрочный тренд (1г)", df[df.index >= df.index[-1] - timedelta(days=365)], 'green')
]

for name, space, color in search_zones:
    # Отрезаем хвост, чтобы не сравнивать с самим собой
    vals, date_found, correlation = find_crypto_fractal(current_data, space.iloc[:-FORECAST])
    if vals is not None:
        fig.add_trace(go.Scatter(
            y=vals,
            name=f"{name} ({date_found})<br>Сходство: {correlation:.2%}",
            line=dict(color=color, width=2, dash='dot'),
            opacity=0.6
        ))

# --- 3. ОФОРМЛЕНИЕ ---
fig.update_layout(
    title=f"<b>Фрактальный прогноз для {TICKER}</b>",
    yaxis_type="log", # ВКЛЮЧАЕМ ЛОГАРИФМИЧЕСКУЮ ШКАЛУ
    xaxis_title="Дни", yaxis_title="Цена (Log Scale)",
    hovermode="x unified", template="plotly_white",
    shapes=[dict(type="line", x0=LOOKBACK-1, x1=LOOKBACK-1,
                 y0=min(current_data)*0.8, y1=max(current_data)*1.2,
                 line=dict(color="gray", dash="dash"))]
)
fig.show()

[*********************100%***********************]  1 of 1 completed


In [14]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta

# --- 1. НАСТРОЙКИ ПОД КРИПТО ---
# Выберите нужный тикер: "BTC-USD" или "ETH-USD"
TICKER = "ETH-USD"
LOOKBACK = 45   # Окно анализа (1.5 месяца)
FORECAST = 30   # Прогноз на месяц вперед

# Загружаем данные (крипта торгуется ежедневно, поэтому данных больше)
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

def find_crypto_fractal(target_series, search_space):
    best_score = float('inf')
    best_idx = -1

    # Используем логарифмическое изменение для крипты (лучше передает характер движения)
    target_log = np.log(target_series)
    target_norm = (target_log - np.mean(target_log)) / np.std(target_log)

    for i in range(len(search_space) - LOOKBACK - FORECAST):
        sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
        sample_log = np.log(sample)
        sample_norm = (sample_log - np.mean(sample_log)) / np.std(sample_log)

        score = np.mean((target_norm - sample_norm)**2)
        if score < best_score:
            best_score = score
            best_idx = i

    if best_idx != -1:
        raw_values = search_space.iloc[best_idx : best_idx + LOOKBACK + FORECAST]['Close'].values
        # Совмещаем по последней цене (Price Action стыковка)
        ratio = target_series[-1] / raw_values[LOOKBACK-1]
        adjusted_values = raw_values * ratio

        corr = np.corrcoef(target_series, raw_values[:LOOKBACK])[0, 1]
        date_str = search_space.index[best_idx].strftime('%Y-%m-%d')
        return adjusted_values, date_str, corr
    return None, None, None

# --- 2. ЗАПУСК ПОИСКА ---
current_data = df['Close'].iloc[-LOOKBACK:].values
fig = go.Figure()

# Основной график BTC/ETH
fig.add_trace(go.Scatter(y=current_data, name=f"ТЕКУЩИЙ {TICKER}",
                         line=dict(color='#F7931A' if "BTC" in TICKER else '#627EEA', width=4)))

# Поиск зон
search_zones = [
    ("Глобальный цикл", df, 'red'),
    ("Цикл после Халвинга (последние 4г)", df[df.index >= df.index[-1] - timedelta(days=4*365)], 'blue'),
    ("Краткосрочный тренд (1г)", df[df.index >= df.index[-1] - timedelta(days=365)], 'green')
]

for name, space, color in search_zones:
    # Отрезаем хвост, чтобы не сравнивать с самим собой
    vals, date_found, correlation = find_crypto_fractal(current_data, space.iloc[:-FORECAST])
    if vals is not None:
        fig.add_trace(go.Scatter(
            y=vals,
            name=f"{name} ({date_found})<br>Сходство: {correlation:.2%}",
            line=dict(color=color, width=2, dash='dot'),
            opacity=0.6
        ))

# --- 3. ОФОРМЛЕНИЕ ---
fig.update_layout(
    title=f"<b>Фрактальный прогноз для {TICKER}</b>",
    yaxis_type="log", # ВКЛЮЧАЕМ ЛОГАРИФМИЧЕСКУЮ ШКАЛУ
    xaxis_title="Дни", yaxis_title="Цена (Log Scale)",
    hovermode="x unified", template="plotly_white",
    shapes=[dict(type="line", x0=LOOKBACK-1, x1=LOOKBACK-1,
                 y0=min(current_data)*0.8, y1=max(current_data)*1.2,
                 line=dict(color="gray", dash="dash"))]
)
fig.show()

[*********************100%***********************]  1 of 1 completed


In [15]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- НАСТРОЙКИ ---
TICKER = "BTC-USD"  # Можно сменить на ETH-USD или ^GSPC
LOOKBACK = 60       # Сколько дней анализируем (прошлое)
FORECAST = 60       # На сколько дней смотрим вперед (будущее)

# Загрузка данных
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

def get_forecast_with_dates(target_series, search_space, current_dates):
    best_score = float('inf')
    best_idx = -1

    # Нормализация для поиска формы
    target_norm = (target_series - np.mean(target_series)) / np.std(target_series)

    for i in range(len(search_space) - LOOKBACK - FORECAST):
        sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
        sample_norm = (sample - np.mean(sample)) / np.std(sample)
        score = np.mean((target_norm - sample_norm)**2)

        if score < best_score:
            best_score = score
            best_idx = i

    if best_idx != -1:
        # Извлекаем исторические значения (включая будущий период)
        raw_values = search_space.iloc[best_idx : best_idx + LOOKBACK + FORECAST]['Close'].values
        # Стыковка по последней цене текущего графика
        ratio = target_series[-1] / raw_values[LOOKBACK-1]
        forecast_values = raw_values * ratio

        # Создаем будущую шкалу времени
        last_date = current_dates[-1]
        future_dates = [last_date + timedelta(days=i) for i in range(1, FORECAST + 1)]
        full_dates = list(current_dates) + future_dates

        return full_dates, forecast_values, search_space.index[best_idx].strftime('%Y-%m-%d')
    return None, None, None

# --- ПОДГОТОВКА ДАННЫХ ---
current_df = df.iloc[-LOOKBACK:]
current_prices = current_df['Close'].values
current_dates = current_df.index

fig = go.Figure()

# 1. Реальный график (черный)
fig.add_trace(go.Scatter(x=current_dates, y=current_prices, name="ТЕКУЩАЯ ЦЕНА",
                         line=dict(color='black', width=4)))

# 2. Поиск и отрисовка циклов
colors = {'Глобальный': 'red', '4 года': 'blue', 'Годовой': 'green'}
zones = [
    ("Глобальный", df),
    ("4 года", df[df.index >= df.index[-1] - timedelta(days=4*365)]),
    ("Годовой", df[df.index >= df.index[-1] - timedelta(days=365)])
]

for name, space in zones:
    dates, values, hist_start = get_forecast_with_dates(current_prices, space.iloc[:-FORECAST], current_dates)

    if values is not None:
        # Рисуем линию цикла
        fig.add_trace(go.Scatter(x=dates, y=values, name=f"Цикл: {name} ({hist_start})",
                                 line=dict(color=colors[name], width=2, dash='dot'), opacity=0.5))

        # Находим экстремумы в ПРЕДСКАЗАННОЙ части (после текущей даты)
        forecast_part = values[LOOKBACK:]
        forecast_dates = dates[LOOKBACK:]

        max_idx = np.argmax(forecast_part)
        min_idx = np.argmin(forecast_part)

        # Добавляем метку ПИКА
        fig.add_annotation(x=forecast_dates[max_idx], y=forecast_part[max_idx],
                           text=f"Пик {forecast_dates[max_idx].strftime('%d.%m')}",
                           showarrow=True, arrowhead=1, bgcolor=colors[name], font=dict(color="white"))

        # Добавляем метку ДНА
        fig.add_annotation(x=forecast_dates[min_idx], y=forecast_part[min_idx],
                           text=f"Дно {forecast_dates[min_idx].strftime('%d.%m')}",
                           showarrow=True, arrowhead=1, bgcolor="black", font=dict(color="white"), ay=40)

# --- ОФОРМЛЕНИЕ ---
fig.update_layout(
    title=f"<b>Прогноз по датам: {TICKER}</b>",
    xaxis_title="Календарная дата",
    yaxis_title="Цена",
    template="plotly_white",
    hovermode="x unified",
    # Вертикальная линия "СЕГОДНЯ"
    shapes=[dict(type="line", x0=current_dates[-1], x1=current_dates[-1],
                 y0=min(current_prices)*0.8, y1=max(current_prices)*1.2,
                 line=dict(color="Gray", width=2, dash="dash"))]
)

# Переключаем на логарифмическую шкалу для крипты
if "USD" in TICKER: fig.update_yaxes(type="log")

fig.show()

[*********************100%***********************]  1 of 1 completed


In [16]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- НАСТРОЙКИ ---
TICKER = "ETH-USD"  # Можно сменить на ETH-USD или ^GSPC
LOOKBACK = 60       # Сколько дней анализируем (прошлое)
FORECAST = 60       # На сколько дней смотрим вперед (будущее)

# Загрузка данных
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

def get_forecast_with_dates(target_series, search_space, current_dates):
    best_score = float('inf')
    best_idx = -1

    # Нормализация для поиска формы
    target_norm = (target_series - np.mean(target_series)) / np.std(target_series)

    for i in range(len(search_space) - LOOKBACK - FORECAST):
        sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
        sample_norm = (sample - np.mean(sample)) / np.std(sample)
        score = np.mean((target_norm - sample_norm)**2)

        if score < best_score:
            best_score = score
            best_idx = i

    if best_idx != -1:
        # Извлекаем исторические значения (включая будущий период)
        raw_values = search_space.iloc[best_idx : best_idx + LOOKBACK + FORECAST]['Close'].values
        # Стыковка по последней цене текущего графика
        ratio = target_series[-1] / raw_values[LOOKBACK-1]
        forecast_values = raw_values * ratio

        # Создаем будущую шкалу времени
        last_date = current_dates[-1]
        future_dates = [last_date + timedelta(days=i) for i in range(1, FORECAST + 1)]
        full_dates = list(current_dates) + future_dates

        return full_dates, forecast_values, search_space.index[best_idx].strftime('%Y-%m-%d')
    return None, None, None

# --- ПОДГОТОВКА ДАННЫХ ---
current_df = df.iloc[-LOOKBACK:]
current_prices = current_df['Close'].values
current_dates = current_df.index

fig = go.Figure()

# 1. Реальный график (черный)
fig.add_trace(go.Scatter(x=current_dates, y=current_prices, name="ТЕКУЩАЯ ЦЕНА",
                         line=dict(color='black', width=4)))

# 2. Поиск и отрисовка циклов
colors = {'Глобальный': 'red', '4 года': 'blue', 'Годовой': 'green'}
zones = [
    ("Глобальный", df),
    ("4 года", df[df.index >= df.index[-1] - timedelta(days=4*365)]),
    ("Годовой", df[df.index >= df.index[-1] - timedelta(days=365)])
]

for name, space in zones:
    dates, values, hist_start = get_forecast_with_dates(current_prices, space.iloc[:-FORECAST], current_dates)

    if values is not None:
        # Рисуем линию цикла
        fig.add_trace(go.Scatter(x=dates, y=values, name=f"Цикл: {name} ({hist_start})",
                                 line=dict(color=colors[name], width=2, dash='dot'), opacity=0.5))

        # Находим экстремумы в ПРЕДСКАЗАННОЙ части (после текущей даты)
        forecast_part = values[LOOKBACK:]
        forecast_dates = dates[LOOKBACK:]

        max_idx = np.argmax(forecast_part)
        min_idx = np.argmin(forecast_part)

        # Добавляем метку ПИКА
        fig.add_annotation(x=forecast_dates[max_idx], y=forecast_part[max_idx],
                           text=f"Пик {forecast_dates[max_idx].strftime('%d.%m')}",
                           showarrow=True, arrowhead=1, bgcolor=colors[name], font=dict(color="white"))

        # Добавляем метку ДНА
        fig.add_annotation(x=forecast_dates[min_idx], y=forecast_part[min_idx],
                           text=f"Дно {forecast_dates[min_idx].strftime('%d.%m')}",
                           showarrow=True, arrowhead=1, bgcolor="black", font=dict(color="white"), ay=40)

# --- ОФОРМЛЕНИЕ ---
fig.update_layout(
    title=f"<b>Прогноз по датам: {TICKER}</b>",
    xaxis_title="Календарная дата",
    yaxis_title="Цена",
    template="plotly_white",
    hovermode="x unified",
    # Вертикальная линия "СЕГОДНЯ"
    shapes=[dict(type="line", x0=current_dates[-1], x1=current_dates[-1],
                 y0=min(current_prices)*0.8, y1=max(current_prices)*1.2,
                 line=dict(color="Gray", width=2, dash="dash"))]
)

# Переключаем на логарифмическую шкалу для крипты
if "USD" in TICKER: fig.update_yaxes(type="log")

fig.show()

[*********************100%***********************]  1 of 1 completed


In [17]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- НАСТРОЙКИ ---
TICKER = "BTC-USD"
LOOKBACK = 60       # Дни для анализа
FORECAST = 45       # Прогноз
TOP_N = 3           # Количество лучших циклов

# Загрузка данных
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

current_df = df.iloc[-LOOKBACK:]
current_prices = current_df['Close'].values
current_dates = current_df.index

# Нормализация текущего движения (Z-score)
target_norm = (current_prices - np.mean(current_prices)) / np.std(current_prices)

results = []

# Поиск всех возможных циклов
search_space = df.iloc[:-FORECAST]
for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
    sample_norm = (sample - np.mean(sample)) / np.std(sample)

    # Считаем MSE
    score = np.mean((target_norm - sample_norm)**2)
    results.append((score, i))

# Сортируем по схожести и берем топ-3
# Фильтруем, чтобы циклы не накладывались друг на друга слишком сильно (минимум 30 дней разницы)
results.sort(key=lambda x: x[0])
best_cycles = []
used_indices = []

for score, idx in results:
    if not any(abs(idx - used_idx) < 30 for used_idx in used_indices):
        best_cycles.append((score, idx))
        used_indices.append(idx)
    if len(best_cycles) == TOP_N: break

# --- ВИЗУАЛИЗАЦИЯ ---
fig = go.Figure()

# Текущая цена
fig.add_trace(go.Scatter(x=np.arange(LOOKBACK), y=current_prices,
                         name="ТЕКУЩАЯ ЦЕНА", line=dict(color='black', width=4)))

colors = ['red', 'blue', 'green']
for i, (score, idx) in enumerate(best_cycles):
    raw_values = search_space.iloc[idx : idx + LOOKBACK + FORECAST]['Close'].values
    # Стыковка по последней цене
    ratio = current_prices[-1] / raw_values[LOOKBACK-1]
    forecast_values = raw_values * ratio

    start_date = search_space.index[idx].strftime('%Y-%m-%d')
    fig.add_trace(go.Scatter(
        y=forecast_values,
        name=f"#{i+1} Цикл ({start_date})",
        line=dict(color=colors[i], width=2, dash='dot'),
        opacity=0.7
    ))

fig.update_layout(
    title=f"<b>Топ-3 похожих цикла для {TICKER}</b>",
    xaxis_title="Дни", yaxis_title="Цена",
    template="plotly_white", hovermode="x unified",
    shapes=[dict(type="line", x0=LOOKBACK-1, x1=LOOKBACK-1, y0=min(current_prices)*0.9, y1=max(current_prices)*1.1, line=dict(color="gray", dash="dash"))]
)
if "USD" in TICKER: fig.update_yaxes(type="log")
fig.show()

[*********************100%***********************]  1 of 1 completed


In [18]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- НАСТРОЙКИ ---
TICKER = "ETH-USD"
LOOKBACK = 60       # Дни для анализа
FORECAST = 45       # Прогноз
TOP_N = 3           # Количество лучших циклов

# Загрузка данных
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

current_df = df.iloc[-LOOKBACK:]
current_prices = current_df['Close'].values
current_dates = current_df.index

# Нормализация текущего движения (Z-score)
target_norm = (current_prices - np.mean(current_prices)) / np.std(current_prices)

results = []

# Поиск всех возможных циклов
search_space = df.iloc[:-FORECAST]
for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
    sample_norm = (sample - np.mean(sample)) / np.std(sample)

    # Считаем MSE
    score = np.mean((target_norm - sample_norm)**2)
    results.append((score, i))

# Сортируем по схожести и берем топ-3
# Фильтруем, чтобы циклы не накладывались друг на друга слишком сильно (минимум 30 дней разницы)
results.sort(key=lambda x: x[0])
best_cycles = []
used_indices = []

for score, idx in results:
    if not any(abs(idx - used_idx) < 30 for used_idx in used_indices):
        best_cycles.append((score, idx))
        used_indices.append(idx)
    if len(best_cycles) == TOP_N: break

# --- ВИЗУАЛИЗАЦИЯ ---
fig = go.Figure()

# Текущая цена
fig.add_trace(go.Scatter(x=np.arange(LOOKBACK), y=current_prices,
                         name="ТЕКУЩАЯ ЦЕНА", line=dict(color='black', width=4)))

colors = ['red', 'blue', 'green']
for i, (score, idx) in enumerate(best_cycles):
    raw_values = search_space.iloc[idx : idx + LOOKBACK + FORECAST]['Close'].values
    # Стыковка по последней цене
    ratio = current_prices[-1] / raw_values[LOOKBACK-1]
    forecast_values = raw_values * ratio

    start_date = search_space.index[idx].strftime('%Y-%m-%d')
    fig.add_trace(go.Scatter(
        y=forecast_values,
        name=f"#{i+1} Цикл ({start_date})",
        line=dict(color=colors[i], width=2, dash='dot'),
        opacity=0.7
    ))

fig.update_layout(
    title=f"<b>Топ-3 похожих цикла для {TICKER}</b>",
    xaxis_title="Дни", yaxis_title="Цена",
    template="plotly_white", hovermode="x unified",
    shapes=[dict(type="line", x0=LOOKBACK-1, x1=LOOKBACK-1, y0=min(current_prices)*0.9, y1=max(current_prices)*1.1, line=dict(color="gray", dash="dash"))]
)
if "USD" in TICKER: fig.update_yaxes(type="log")
fig.show()

[*********************100%***********************]  1 of 1 completed


In [19]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- НАСТРОЙКИ ---
TICKER = "SOL-USD"
LOOKBACK = 60       # Дни для анализа
FORECAST = 45       # Прогноз
TOP_N = 3           # Количество лучших циклов

# Загрузка данных
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

current_df = df.iloc[-LOOKBACK:]
current_prices = current_df['Close'].values
current_dates = current_df.index

# Нормализация текущего движения (Z-score)
target_norm = (current_prices - np.mean(current_prices)) / np.std(current_prices)

results = []

# Поиск всех возможных циклов
search_space = df.iloc[:-FORECAST]
for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
    sample_norm = (sample - np.mean(sample)) / np.std(sample)

    # Считаем MSE
    score = np.mean((target_norm - sample_norm)**2)
    results.append((score, i))

# Сортируем по схожести и берем топ-3
# Фильтруем, чтобы циклы не накладывались друг на друга слишком сильно (минимум 30 дней разницы)
results.sort(key=lambda x: x[0])
best_cycles = []
used_indices = []

for score, idx in results:
    if not any(abs(idx - used_idx) < 30 for used_idx in used_indices):
        best_cycles.append((score, idx))
        used_indices.append(idx)
    if len(best_cycles) == TOP_N: break

# --- ВИЗУАЛИЗАЦИЯ ---
fig = go.Figure()

# Текущая цена
fig.add_trace(go.Scatter(x=np.arange(LOOKBACK), y=current_prices,
                         name="ТЕКУЩАЯ ЦЕНА", line=dict(color='black', width=4)))

colors = ['red', 'blue', 'green']
for i, (score, idx) in enumerate(best_cycles):
    raw_values = search_space.iloc[idx : idx + LOOKBACK + FORECAST]['Close'].values
    # Стыковка по последней цене
    ratio = current_prices[-1] / raw_values[LOOKBACK-1]
    forecast_values = raw_values * ratio

    start_date = search_space.index[idx].strftime('%Y-%m-%d')
    fig.add_trace(go.Scatter(
        y=forecast_values,
        name=f"#{i+1} Цикл ({start_date})",
        line=dict(color=colors[i], width=2, dash='dot'),
        opacity=0.7
    ))

fig.update_layout(
    title=f"<b>Топ-3 похожих цикла для {TICKER}</b>",
    xaxis_title="Дни", yaxis_title="Цена",
    template="plotly_white", hovermode="x unified",
    shapes=[dict(type="line", x0=LOOKBACK-1, x1=LOOKBACK-1, y0=min(current_prices)*0.9, y1=max(current_prices)*1.1, line=dict(color="gray", dash="dash"))]
)
if "USD" in TICKER: fig.update_yaxes(type="log")
fig.show()

[*********************100%***********************]  1 of 1 completed


In [20]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- 1. НАСТРОЙКИ ---
TICKER = "BTC-USD"  # Можно заменить на ETH-USD
LOOKBACK = 90       # Анализируем последние 3 месяца
FORECAST = 30       # Прогноз на 1 месяц

# Загрузка данных
df = yf.download(TICKER, start="2010-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

# Текущее движение
current_prices = df['Close'].iloc[-LOOKBACK:].values
current_dates = df.index[-LOOKBACK:]

# --- 2. ПОИСК АБСОЛЮТНО ЛУЧШЕГО ЦИКЛА ---
best_corr = -1
best_idx = -1

# Проходим по всей истории (исключая будущий период прогноза)
search_space = df.iloc[:-FORECAST]
for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values

    # Считаем корреляцию Пирсона (насколько формы идентичны)
    corr = np.corrcoef(current_prices, sample)[0, 1]

    if corr > best_corr:
        best_corr = corr
        best_idx = i

# --- 3. ПОДГОТОВКА И ВИЗУАЛИЗАЦИЯ ---
best_cycle_data = df.iloc[best_idx : best_idx + LOOKBACK + FORECAST]
best_date = best_cycle_data.index[0].strftime('%Y-%m-%d')

# Стыкуем исторические данные с текущей ценой
ratio = current_prices[-1] / best_cycle_data['Close'].values[LOOKBACK-1]
projected_prices = best_cycle_data['Close'].values * ratio

# Создаем шкалу времени для прогноза
future_dates = [current_dates[-1] + timedelta(days=i) for i in range(1, FORECAST + 1)]
all_dates = list(current_dates) + future_dates

fig = go.Figure()

# Текущая цена
fig.add_trace(go.Scatter(x=current_dates, y=current_prices, name="Текущая цена", line=dict(color='black', width=4)))

# Лучший найденный цикл
fig.add_trace(go.Scatter(x=all_dates, y=projected_prices, name=f"Лучший цикл ({best_date})",
                         line=dict(color='red', width=2, dash='dot')))

# Индикатор прогноза
fig.add_vrect(x0=current_dates[-1], x1=all_dates[-1], fillcolor="green", opacity=0.1,
              layer="below", line_width=0, annotation_text="ПРОГНОЗ")

fig.update_layout(
    title=f"<b>Лучший исторический цикл для {TICKER}</b><br><sup>Сходство (Correlation): {best_corr:.2%} | Начало цикла: {best_date}</sup>",
    xaxis_title="Дата", yaxis_title="Цена",
    template="plotly_white", hovermode="x unified"
)

if "USD" in TICKER: fig.update_yaxes(type="log")
fig.show()

[*********************100%***********************]  1 of 1 completed


In [21]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- 1. НАСТРОЙКИ ---
TICKER = "ETH-USD"  # Можно заменить на ETH-USD
LOOKBACK = 90       # Анализируем последние 3 месяца
FORECAST = 30       # Прогноз на 1 месяц

# Загрузка данных
df = yf.download(TICKER, start="2010-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

# Текущее движение
current_prices = df['Close'].iloc[-LOOKBACK:].values
current_dates = df.index[-LOOKBACK:]

# --- 2. ПОИСК АБСОЛЮТНО ЛУЧШЕГО ЦИКЛА ---
best_corr = -1
best_idx = -1

# Проходим по всей истории (исключая будущий период прогноза)
search_space = df.iloc[:-FORECAST]
for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values

    # Считаем корреляцию Пирсона (насколько формы идентичны)
    corr = np.corrcoef(current_prices, sample)[0, 1]

    if corr > best_corr:
        best_corr = corr
        best_idx = i

# --- 3. ПОДГОТОВКА И ВИЗУАЛИЗАЦИЯ ---
best_cycle_data = df.iloc[best_idx : best_idx + LOOKBACK + FORECAST]
best_date = best_cycle_data.index[0].strftime('%Y-%m-%d')

# Стыкуем исторические данные с текущей ценой
ratio = current_prices[-1] / best_cycle_data['Close'].values[LOOKBACK-1]
projected_prices = best_cycle_data['Close'].values * ratio

# Создаем шкалу времени для прогноза
future_dates = [current_dates[-1] + timedelta(days=i) for i in range(1, FORECAST + 1)]
all_dates = list(current_dates) + future_dates

fig = go.Figure()

# Текущая цена
fig.add_trace(go.Scatter(x=current_dates, y=current_prices, name="Текущая цена", line=dict(color='black', width=4)))

# Лучший найденный цикл
fig.add_trace(go.Scatter(x=all_dates, y=projected_prices, name=f"Лучший цикл ({best_date})",
                         line=dict(color='red', width=2, dash='dot')))

# Индикатор прогноза
fig.add_vrect(x0=current_dates[-1], x1=all_dates[-1], fillcolor="green", opacity=0.1,
              layer="below", line_width=0, annotation_text="ПРОГНОЗ")

fig.update_layout(
    title=f"<b>Лучший исторический цикл для {TICKER}</b><br><sup>Сходство (Correlation): {best_corr:.2%} | Начало цикла: {best_date}</sup>",
    xaxis_title="Дата", yaxis_title="Цена",
    template="plotly_white", hovermode="x unified"
)

if "USD" in TICKER: fig.update_yaxes(type="log")
fig.show()

[*********************100%***********************]  1 of 1 completed


In [22]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- 1. НАСТРОЙКИ ---
TICKER = "ETH-USD"
LOOKBACK_RANGE = range(30, 121, 5) # Ищем оптимальное окно от 30 до 120 дней
FORECAST = 30

# Загрузка данных
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

# --- 2. ГЛУБИННЫЙ АВТОПОДБОР (GRID SEARCH) ---
overall_best_corr = -1
overall_best_idx = -1
overall_best_lb = -1

print("Запуск глубинного поиска лучшего окна...")

for lb in LOOKBACK_RANGE:
    current_prices = df['Close'].iloc[-lb:].values
    search_space = df.iloc[:-FORECAST - lb]

    # Поиск лучшего совпадения для данного окна
    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        corr = np.corrcoef(current_prices, sample)[0, 1]

        if corr > overall_best_corr:
            overall_best_corr = corr
            overall_best_idx = i
            overall_best_lb = lb

print(f"\nОптимальное окно (LOOKBACK) найдено: {overall_best_lb} дней")
print(f"Максимальная корреляция: {overall_best_corr:.2%}")

# --- 3. ВИЗУАЛИЗАЦИЯ ЛУЧШЕГО ИЗ ЛУЧШИХ ---
best_cycle_data = df.iloc[overall_best_idx : overall_best_idx + overall_best_lb + FORECAST]
best_date = best_cycle_data.index[0].strftime('%Y-%m-%d')
current_prices_final = df['Close'].iloc[-overall_best_lb:].values
current_dates_final = df.index[-overall_best_lb:]

# Стыковка
ratio = current_prices_final[-1] / best_cycle_data['Close'].values[overall_best_lb-1]
projected_prices = best_cycle_data['Close'].values * ratio

future_dates = [current_dates_final[-1] + timedelta(days=i) for i in range(1, FORECAST + 1)]
all_dates = list(current_dates_final) + future_dates

fig = go.Figure()
fig.add_trace(go.Scatter(x=current_dates_final, y=current_prices_final, name="Текущая цена", line=dict(color='black', width=4)))
fig.add_trace(go.Scatter(x=all_dates, y=projected_prices, name=f"Лучший фрактал ({best_date}, LB: {overall_best_lb})", line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=current_dates_final[-1], x1=all_dates[-1], fillcolor="blue", opacity=0.05, annotation_text="ЗОНА ПРОГНОЗА")

fig.update_layout(
    title=f"<b>Глубинный автоподбор фрактала для {TICKER}</b><br><sup>Лучшее окно: {overall_best_lb} дн. | Корреляция: {overall_best_corr:.2%}</sup>",
    template="plotly_white", hovermode="x unified"
)
if "USD" in TICKER: fig.update_yaxes(type="log")
fig.show()

[*********************100%***********************]  1 of 1 completed


Запуск глубинного поиска лучшего окна...

Оптимальное окно (LOOKBACK) найдено: 45 дней
Максимальная корреляция: 91.31%


In [23]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- 1. НАСТРОЙКИ ---
TICKER = "SOL-USD"
LOOKBACK_RANGE = range(30, 121, 5) # Ищем оптимальное окно от 30 до 120 дней
FORECAST = 30

# Загрузка данных
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

# --- 2. ГЛУБИННЫЙ АВТОПОДБОР (GRID SEARCH) ---
overall_best_corr = -1
overall_best_idx = -1
overall_best_lb = -1

print("Запуск глубинного поиска лучшего окна...")

for lb in LOOKBACK_RANGE:
    current_prices = df['Close'].iloc[-lb:].values
    search_space = df.iloc[:-FORECAST - lb]

    # Поиск лучшего совпадения для данного окна
    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        corr = np.corrcoef(current_prices, sample)[0, 1]

        if corr > overall_best_corr:
            overall_best_corr = corr
            overall_best_idx = i
            overall_best_lb = lb

print(f"\nОптимальное окно (LOOKBACK) найдено: {overall_best_lb} дней")
print(f"Максимальная корреляция: {overall_best_corr:.2%}")

# --- 3. ВИЗУАЛИЗАЦИЯ ЛУЧШЕГО ИЗ ЛУЧШИХ ---
best_cycle_data = df.iloc[overall_best_idx : overall_best_idx + overall_best_lb + FORECAST]
best_date = best_cycle_data.index[0].strftime('%Y-%m-%d')
current_prices_final = df['Close'].iloc[-overall_best_lb:].values
current_dates_final = df.index[-overall_best_lb:]

# Стыковка
ratio = current_prices_final[-1] / best_cycle_data['Close'].values[overall_best_lb-1]
projected_prices = best_cycle_data['Close'].values * ratio

future_dates = [current_dates_final[-1] + timedelta(days=i) for i in range(1, FORECAST + 1)]
all_dates = list(current_dates_final) + future_dates

fig = go.Figure()
fig.add_trace(go.Scatter(x=current_dates_final, y=current_prices_final, name="Текущая цена", line=dict(color='black', width=4)))
fig.add_trace(go.Scatter(x=all_dates, y=projected_prices, name=f"Лучший фрактал ({best_date}, LB: {overall_best_lb})", line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=current_dates_final[-1], x1=all_dates[-1], fillcolor="blue", opacity=0.05, annotation_text="ЗОНА ПРОГНОЗА")

fig.update_layout(
    title=f"<b>Глубинный автоподбор фрактала для {TICKER}</b><br><sup>Лучшее окно: {overall_best_lb} дн. | Корреляция: {overall_best_corr:.2%}</sup>",
    template="plotly_white", hovermode="x unified"
)
if "USD" in TICKER: fig.update_yaxes(type="log")
fig.show()

[*********************100%***********************]  1 of 1 completed


Запуск глубинного поиска лучшего окна...

Оптимальное окно (LOOKBACK) найдено: 120 дней
Максимальная корреляция: 94.14%


In [24]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- FINAL CONSOLIDATED PARAMETERS ---
TICKER = "ETH-USD"
OPTIMIZED_LB = 45 # Derived from previous Grid Search
FORECAST = 45

# Load fresh data
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

def find_fractal(target_prices, search_space, lookback, forecast, min_corr=0.0):
    best_corr = -1
    best_idx = -1
    for i in range(len(search_space) - lookback - forecast):
        sample = search_space.iloc[i : i + lookback]['Close'].values
        corr = np.corrcoef(target_prices, sample)[0, 1]
        if corr > best_corr:
            best_corr = corr
            best_idx = i

    if best_idx != -1 and best_corr >= min_corr:
        raw = search_space.iloc[best_idx : best_idx + lookback + forecast]['Close'].values
        ratio = target_prices[-1] / raw[lookback-1]
        return raw * ratio, best_corr, search_space.index[best_idx].strftime('%Y-%m-%d')
    return None, None, None

# Prepare target series
target_opt = df['Close'].iloc[-OPTIMIZED_LB:].values
target_std = df['Close'].iloc[-60:].values # Standard 60d for other comparisons

fig = go.Figure()

# 1. ACTUAL PRICE
fig.add_trace(go.Scatter(x=np.arange(60), y=target_std, name="CURRENT PRICE", line=dict(color='black', width=5)))

# CONFIGURATIONS FOR MULTI-FORECAST
scenarios = [
    ("Optimized (45d)", df, OPTIMIZED_LB, 0.0, 'red'),
    ("Global (60d)", df, 60, 0.0, 'blue'),
    ("Halving (4yr)", df[df.index >= df.index[-1] - timedelta(days=4*365)], 60, 0.0, 'green'),
    ("Annual (1yr)", df[df.index >= df.index[-1] - timedelta(days=365)], 60, 0.0, 'orange'),
    ("High Confidence (>90%)", df, 60, 0.90, 'purple')
]

for name, space, lb, min_c, color in scenarios:
    search_data = space.iloc[:-FORECAST]
    current_target = df['Close'].iloc[-lb:].values
    vals, corr, d_start = find_fractal(current_target, search_data, lb, FORECAST, min_c)

    if vals is not None:
        # Align x-axis so 0 is start of lookback relative to current 60d window
        x_offset = 60 - lb
        fig.add_trace(go.Scatter(
            x=np.arange(x_offset, x_offset + lb + FORECAST),
            y=vals,
            name=f"{name} (Corr: {corr:.1%})",
            line=dict(color=color, width=2, dash='dot'),
            opacity=0.7
        ))

fig.add_vrect(x0=59, x1=59+FORECAST, fillcolor="gray", opacity=0.1, annotation_text="FORECAST")
fig.update_layout(
    title=f"<b>Final Consolidated Fractal Research: {TICKER}</b>",
    xaxis_title="Relative Days", yaxis_title="Price (USD)",
    yaxis_type="log", template="plotly_white", hovermode="x unified"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


In [25]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- ПАРАМЕТРЫ КОНСОЛИДАЦИИ ---
TICKER = "ETH-USD"
LOOKBACK_OPT = 45
LOOKBACK_STD = 60
FORECAST = 45

df = yf.download(TICKER, start='2014-01-01', auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

def get_fractal(target, search_space, lb, fc, method='mse'):
    best_score = float('inf') if method == 'mse' else -1
    best_idx = -1

    t_norm = (target - np.mean(target)) / np.std(target)

    for i in range(len(search_space) - lb - fc):
        sample = search_space.iloc[i : i + lb]['Close'].values
        s_norm = (sample - np.mean(sample)) / np.std(sample)

        if method == 'mse':
            score = np.mean((t_norm - s_norm)**2)
            if score < best_score:
                best_score, best_idx = score, i
        else: # correlation
            score = np.corrcoef(target, sample)[0, 1]
            if score > best_score:
                best_score, best_idx = score, i

    if best_idx != -1:
        raw = search_space.iloc[best_idx : best_idx + lb + fc]['Close'].values
        ratio = target[-1] / raw[lb-1]
        return raw * ratio, best_score, search_space.index[best_idx]
    return None, None, None

# Подготовка данных
current_prices = df['Close'].iloc[-LOOKBACK_STD:].values
search_area = df.iloc[:-FORECAST]

fig = go.Figure()

# 1. ТЕКУЩАЯ ЦЕНА
fig.add_trace(go.Scatter(x=np.arange(LOOKBACK_STD), y=current_prices, name="ФАКТ (ETH)", line=dict(color='black', width=5)))

# Сценарии
scenarios = [
    ('Оптимальное окно (45д)', LOOKBACK_OPT, 'mse', 'red'),
    ('Зигзаг/MSE (60д)', LOOKBACK_STD, 'mse', 'blue'),
    ('Корреляция (60д)', LOOKBACK_STD, 'corr', 'green'),
    ('Глобальный поиск', 90, 'mse', 'orange'),
    ('High Confidence (>90%)', 60, 'corr', 'purple')
]

for name, lb, method, color in scenarios:
    target = df['Close'].iloc[-lb:].values
    vals, score, d_start = get_fractal(target, search_area, lb, FORECAST, method)

    if vals is not None:
        # Если это High Confidence, проверяем порог
        if 'Confidence' in name and score < 0.90: continue

        offset = LOOKBACK_STD - lb
        label = f"{name} ({score:.2f})" if method == 'mse' else f"{name} ({score:.1%})"

        fig.add_trace(go.Scatter(
            x=np.arange(offset, offset + lb + FORECAST),
            y=vals,
            name=label,
            line=dict(color=color, width=2, dash='dot'),
            opacity=0.6
        ))

fig.add_vrect(x0=LOOKBACK_STD-1, x1=LOOKBACK_STD+FORECAST-1, fillcolor="gray", opacity=0.1, annotation_text="ЗОНА ПРОГНОЗА")
fig.update_layout(
    title=f"<b>Финальный консолидированный прогноз {TICKER}</b>",
    xaxis_title="Относительные дни", yaxis_title="Цена (Log)",
    yaxis_type="log", template="plotly_white", hovermode="x unified"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


# FINAL CONSOLIDATED RESEARCH
This block synthesizes the three primary forecasting methods: Optimized Lookback, Global Historical Search, and High-Confidence Correlation matching.

In [26]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS ---
TICKER = "ETH-USD"
OPTIMIZED_LB = 45
STD_LB = 60
FORECAST = 45

df = yf.download(TICKER, start='2014-01-01', auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

def find_fractal_logic(target, search_space, lb, fc, mode='mse'):
    best_val = float('inf') if mode == 'mse' else -1
    best_idx = -1
    t_norm = (target - np.mean(target)) / np.std(target)

    for i in range(len(search_space) - lb - fc):
        sample = search_space.iloc[i : i + lb]['Close'].values
        if mode == 'mse':
            s_norm = (sample - np.mean(sample)) / np.std(sample)
            score = np.mean((t_norm - s_norm)**2)
            if score < best_val: best_val, best_idx = score, i
        else:
            score = np.corrcoef(target, sample)[0, 1]
            if score > best_val: best_val, best_idx = score, i

    if best_idx != -1:
        raw = search_space.iloc[best_idx : best_idx + lb + fc]['Close'].values
        ratio = target[-1] / raw[lb-1]
        return raw * ratio, best_val, search_space.index[best_idx]
    return None, None, None

# Data Prep
current_prices = df['Close'].iloc[-STD_LB:].values
search_area = df.iloc[:-FORECAST]
fig = go.Figure()

# 1. ACTUAL PRICE
fig.add_trace(go.Scatter(x=np.arange(STD_LB), y=current_prices, name="ACTUAL ETH", line=dict(color='black', width=5)))

# 2. SCENARIOS
# A: Optimized Window (45d)
target_opt = df['Close'].iloc[-OPTIMIZED_LB:].values
vals_opt, score_opt, date_opt = find_fractal_logic(target_opt, search_area, OPTIMIZED_LB, FORECAST, 'mse')
if vals_opt is not None:
    fig.add_trace(go.Scatter(x=np.arange(STD_LB-OPTIMIZED_LB, STD_LB+FORECAST), y=vals_opt,
                             name=f"Optimized (LB:45, MSE:{score_opt:.2f})", line=dict(color='red', width=2, dash='dot')))

# B: Global Search (Standard 60d)
vals_gb, score_gb, date_gb = find_fractal_logic(current_prices, search_area, STD_LB, FORECAST, 'mse')
if vals_gb is not None:
    fig.add_trace(go.Scatter(x=np.arange(STD_LB+FORECAST), y=vals_gb,
                             name=f"Global (LB:60, MSE:{score_gb:.2f})", line=dict(color='orange', width=2, dash='dot')))

# C: High Confidence Correlation (>90%)
vals_hc, score_hc, date_hc = find_fractal_logic(current_prices, search_area, STD_LB, FORECAST, 'corr')
if vals_hc is not None and score_hc >= 0.90:
    fig.add_trace(go.Scatter(x=np.arange(STD_LB+FORECAST), y=vals_hc,
                             name=f"High Confidence ({score_hc:.1%})", line=dict(color='purple', width=3)))

# Layout
fig.add_vrect(x0=STD_LB-1, x1=STD_LB+FORECAST-1, fillcolor="gray", opacity=0.1, annotation_text="FORECAST ZONE")
fig.update_layout(title=f"<b>Final Consolidated Forecast: {TICKER}</b>", yaxis_type="log",
                  xaxis_title="Relative Days", yaxis_title="Price (USD)", template="plotly_white", hovermode="x unified")
fig.show()

[*********************100%***********************]  1 of 1 completed


In [27]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS ---
TICKER = "SOL-USD"
OPTIMIZED_LB = 45
STD_LB = 60
FORECAST = 45

df = yf.download(TICKER, start='2014-01-01', auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

def find_fractal_logic(target, search_space, lb, fc, mode='mse'):
    best_val = float('inf') if mode == 'mse' else -1
    best_idx = -1
    t_norm = (target - np.mean(target)) / np.std(target)

    for i in range(len(search_space) - lb - fc):
        sample = search_space.iloc[i : i + lb]['Close'].values
        if mode == 'mse':
            s_norm = (sample - np.mean(sample)) / np.std(sample)
            score = np.mean((t_norm - s_norm)**2)
            if score < best_val: best_val, best_idx = score, i
        else:
            score = np.corrcoef(target, sample)[0, 1]
            if score > best_val: best_val, best_idx = score, i

    if best_idx != -1:
        raw = search_space.iloc[best_idx : best_idx + lb + fc]['Close'].values
        ratio = target[-1] / raw[lb-1]
        return raw * ratio, best_val, search_space.index[best_idx]
    return None, None, None

# Data Prep
current_prices = df['Close'].iloc[-STD_LB:].values
search_area = df.iloc[:-FORECAST]
fig = go.Figure()

# 1. ACTUAL PRICE
fig.add_trace(go.Scatter(x=np.arange(STD_LB), y=current_prices, name="ACTUAL ETH", line=dict(color='black', width=5)))

# 2. SCENARIOS
# A: Optimized Window (45d)
target_opt = df['Close'].iloc[-OPTIMIZED_LB:].values
vals_opt, score_opt, date_opt = find_fractal_logic(target_opt, search_area, OPTIMIZED_LB, FORECAST, 'mse')
if vals_opt is not None:
    fig.add_trace(go.Scatter(x=np.arange(STD_LB-OPTIMIZED_LB, STD_LB+FORECAST), y=vals_opt,
                             name=f"Optimized (LB:45, MSE:{score_opt:.2f})", line=dict(color='red', width=2, dash='dot')))

# B: Global Search (Standard 60d)
vals_gb, score_gb, date_gb = find_fractal_logic(current_prices, search_area, STD_LB, FORECAST, 'mse')
if vals_gb is not None:
    fig.add_trace(go.Scatter(x=np.arange(STD_LB+FORECAST), y=vals_gb,
                             name=f"Global (LB:60, MSE:{score_gb:.2f})", line=dict(color='orange', width=2, dash='dot')))

# C: High Confidence Correlation (>90%)
vals_hc, score_hc, date_hc = find_fractal_logic(current_prices, search_area, STD_LB, FORECAST, 'corr')
if vals_hc is not None and score_hc >= 0.90:
    fig.add_trace(go.Scatter(x=np.arange(STD_LB+FORECAST), y=vals_hc,
                             name=f"High Confidence ({score_hc:.1%})", line=dict(color='purple', width=3)))

# Layout
fig.add_vrect(x0=STD_LB-1, x1=STD_LB+FORECAST-1, fillcolor="gray", opacity=0.1, annotation_text="FORECAST ZONE")
fig.update_layout(title=f"<b>Final Consolidated Forecast: {TICKER}</b>", yaxis_type="log",
                  xaxis_title="Relative Days", yaxis_title="Price (USD)", template="plotly_white", hovermode="x unified")
fig.show()

[*********************100%***********************]  1 of 1 completed


In [28]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS ---
TICKER = "ETH-USD"
LOOKBACK = 45
FORECAST = 45
TOP_N = 3

# Load data
df = yf.download(TICKER, start='2014-01-01', auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

# Current pattern
target = df['Close'].iloc[-LOOKBACK:].values
target_norm = (target - np.mean(target)) / np.std(target)

# Search all history
search_space = df.iloc[:-FORECAST]
results = []

for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
    sample_norm = (sample - np.mean(sample)) / np.std(sample)

    mse = np.mean((target_norm - sample_norm)**2)
    results.append((mse, i))

# Filter for distinct non-overlapping cycles (min 30 days apart)
results.sort(key=lambda x: x[0])
best_matches = []
used_indices = []

for mse, idx in results:
    if not any(abs(idx - u) < 30 for u in used_indices):
        best_matches.append((mse, idx))
        used_indices.append(idx)
    if len(best_matches) == TOP_N: break

# Visualization
fig = go.Figure()

# Actual Price
fig.add_trace(go.Scatter(x=np.arange(LOOKBACK), y=target, name="CURRENT ETH", line=dict(color='black', width=5)))

colors = ['red', 'blue', 'green']
for i, (mse, idx) in enumerate(best_matches):
    raw = df.iloc[idx : idx + LOOKBACK + FORECAST]['Close'].values
    ratio = target[-1] / raw[LOOKBACK-1]
    vals = raw * ratio
    start_date = df.index[idx].strftime('%Y-%m-%d')

    fig.add_trace(go.Scatter(
        x=np.arange(LOOKBACK + FORECAST),
        y=vals,
        name=f"Match #{i+1} ({start_date}) MSE: {mse:.3f}",
        line=dict(color=colors[i], width=2, dash='dot'),
        opacity=0.7
    ))

fig.add_vrect(x0=LOOKBACK-1, x1=LOOKBACK+FORECAST-1, fillcolor="gray", opacity=0.1, annotation_text="PROJECTION")
fig.update_layout(
    title=f"<b>Top {TOP_N} Best Historical Matches for {TICKER} (Lowest MSE)</b>",
    yaxis_type="log",
    xaxis_title="Relative Days",
    yaxis_title="Price (USD)",
    template="plotly_white",
    hovermode="x unified"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


In [29]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- НАСТРОЙКИ ---
TICKERS = ["BTC-USD", "ETH-USD"]
LB_RANGE = range(10, 181, 5) # Проверяем окна от 10 до 180 дней

# 1. Загрузка данных
print("Загрузка данных...")
df = yf.download(TICKERS, period="2y", interval="1d")['Close']
df = df.dropna()

# 2. Расчет корреляций для разных окон
lb_results = []
for lb in LB_RANGE:
    # Берем последние 'lb' дней
    recent_data = df.iloc[-lb:]
    correlation = recent_data[TICKERS[0]].corr(recent_data[TICKERS[1]])
    lb_results.append({"Lookback": lb, "Correlation": correlation})

lb_df = pd.DataFrame(lb_results)

# Находим лучший Lookback
best_lb = lb_df.loc[lb_df['Correlation'].idxmax()]

# 3. Визуализация
fig = go.Figure()
fig.add_trace(go.Scatter(x=lb_df['Lookback'], y=lb_df['Correlation'],
                         mode='lines+markers', name='Correlation Strength'))

fig.add_annotation(x=best_lb['Lookback'], y=best_lb['Correlation'],
            text=f"BEST: {int(best_lb['Lookback'])} days ({best_lb['Correlation']:.2%})",
            showarrow=True, arrowhead=1, bgcolor="green", font=dict(color="white"))

fig.update_layout(
    title="<b>Оптимизация окна LOOKBACK: BTC vs ETH</b><br><sup>Поиск периода с максимальной синхронностью активов</sup>",
    xaxis_title="Размер окна (дни)",
    yaxis_title="Корреляция Пирсона",
    template="plotly_white"
)
fig.show()

print(f"Идеальное окно для анализа: {int(best_lb['Lookback'])} дней.")

/tmp/ipykernel_23466/2114008189.py:12: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  2 of 2 completed

Загрузка данных...


Идеальное окно для анализа: 175 дней.


In [30]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS ---
TICKERS = ["BTC-USD", "SOL-USD"]
LOOKBACK = 180  # User requested 180-day window
FORECAST = 60

# Ensure we use the full history available for best fractal matching
df_all = yf.download(TICKERS, start="2014-01-01", auto_adjust=True)['Close']
df_all = df_all.dropna()

def get_best_fractal(ticker_name, data, lb, fc):
    # Get current pattern
    current_pattern = data[ticker_name].iloc[-lb:].values
    target_norm = (current_pattern - np.mean(current_pattern)) / np.std(current_pattern)

    # Define search space (all history excluding the forecast period from the end)
    search_space = data[ticker_name].iloc[:-fc]
    best_score = float('inf')
    best_idx = -1

    # Slide window
    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb].values
        sample_norm = (sample - np.mean(sample)) / np.std(sample)
        score = np.mean((target_norm - sample_norm)**2)

        if score < best_score:
            best_score = score
            best_idx = i

    if best_idx != -1:
        # Extract found segment plus its future
        raw_segment = data[ticker_name].iloc[best_idx : best_idx + lb + fc].values
        # Anchor to current price
        ratio = current_pattern[-1] / raw_segment[lb-1]
        projected = raw_segment * ratio
        start_date = data.index[best_idx].strftime('%Y-%m-%d')
        return projected, start_date, best_score
    return None, None, None

# Run and Plot
for ticker in TICKERS:
    proj_vals, s_date, score = get_best_fractal(ticker, df_all, LOOKBACK, FORECAST)

    if proj_vals is not None:
        fig = go.Figure()

        # Current Price
        fig.add_trace(go.Scatter(y=df_all[ticker].iloc[-LOOKBACK:].values,
                                 name=f"Current {ticker}", line=dict(color='black', width=4)))

        # Fractal Projection
        fig.add_trace(go.Scatter(y=proj_vals,
                                 name=f"Fractal from {s_date} (MSE: {score:.3f})",
                                 line=dict(color='red', width=2, dash='dot')))

        fig.add_vrect(x0=LOOKBACK-1, x1=LOOKBACK+FORECAST-1, fillcolor="gray", opacity=0.1,
                      annotation_text="FORECAST")

        fig.update_layout(title=f"<b>Fractal Forecast: {ticker} (180d Lookback)</b>",
                          xaxis_title="Days (Relative)", yaxis_title="Price (USD)",
                          template="plotly_white", yaxis_type="log")
        fig.show()

[*********************100%***********************]  2 of 2 completed


In [31]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- НАСТРОЙКИ ---
TICKERS = ["BTC-USD", "SOL-USD"]
LB_RANGE = range(10, 181, 5)

# 1. Загрузка данных
print("Загрузка данных...")
df_sol = yf.download(TICKERS, period="2y", interval="1d")['Close']
df_sol = df_sol.dropna()

# 2. Расчет корреляций
lb_results_sol = []
for lb in LB_RANGE:
    recent_data = df_sol.iloc[-lb:]
    correlation = recent_data[TICKERS[0]].corr(recent_data[TICKERS[1]])
    lb_results_sol.append({"Lookback": lb, "Correlation": correlation})

lb_df_sol = pd.DataFrame(lb_results_sol)
best_lb_sol = lb_df_sol.loc[lb_df_sol['Correlation'].idxmax()]

# 3. Визуализация
fig = go.Figure()
fig.add_trace(go.Scatter(x=lb_df_sol['Lookback'], y=lb_df_sol['Correlation'],
                         mode='lines+markers', name='BTC vs SOL'))

fig.add_annotation(x=best_lb_sol['Lookback'], y=best_lb_sol['Correlation'],
            text=f"BEST: {int(best_lb_sol['Lookback'])} days ({best_lb_sol['Correlation']:.2%})",
            showarrow=True, arrowhead=1, bgcolor="orange", font=dict(color="white"))

fig.update_layout(
    title="<b>Оптимизация окна LOOKBACK: BTC vs SOL</b>",
    xaxis_title="Размер окна (дни)",
    yaxis_title="Корреляция Пирсона",
    template="plotly_white"
)
fig.show()

print(f"Идеальное окно для анализа BTC vs SOL: {int(best_lb_sol['Lookback'])} дней.")

/tmp/ipykernel_23466/499214286.py:12: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  2 of 2 completed

Загрузка данных...


Идеальное окно для анализа BTC vs SOL: 180 дней.


In [32]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- НАСТРОЙКИ ---
TICKERS = ["BTC-USD", "ETH-USD"]
LOOKBACK = 175  # Оптимизированное окно
FORECAST = 60   # Прогноз на 60 дней

# Загрузка данных (всей доступной истории)
df_all = yf.download(TICKERS, start="2014-01-01", auto_adjust=True)['Close']
df_all = df_all.dropna()

def get_best_fractal(ticker_name, data, lb, fc):
    current_pattern = data[ticker_name].iloc[-lb:].values
    target_norm = (current_pattern - np.mean(current_pattern)) / np.std(current_pattern)

    search_space = data[ticker_name].iloc[:-fc]
    best_score = float('inf')
    best_idx = -1

    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb].values
        sample_norm = (sample - np.mean(sample)) / np.std(sample)
        score = np.mean((target_norm - sample_norm)**2)

        if score < best_score:
            best_score = score
            best_idx = i

    if best_idx != -1:
        # Извлекаем исторический отрезок + будущее
        raw_segment = data[ticker_name].iloc[best_idx : best_idx + lb + fc].values
        # Стыковка
        ratio = current_pattern[-1] / raw_segment[lb-1]
        projected = raw_segment * ratio
        start_date = data.index[best_idx].strftime('%Y-%m-%d')
        return projected, start_date, best_score
    return None, None, None

# Визуализация для каждого тикера
for ticker in TICKERS:
    proj_vals, s_date, score = get_best_fractal(ticker, df_all, LOOKBACK, FORECAST)

    if proj_vals is not None:
        fig = go.Figure()

        # Текущая цена
        fig.add_trace(go.Scatter(y=df_all[ticker].iloc[-LOOKBACK:].values,
                                 name=f"Текущий {ticker}", line=dict(color='black', width=4)))

        # Фрактал
        fig.add_trace(go.Scatter(y=proj_vals,
                                 name=f"Фрактал от {s_date} (MSE: {score:.3f})",
                                 line=dict(color='red', width=2, dash='dot')))

        fig.add_vrect(x0=LOOKBACK-1, x1=LOOKBACK+FORECAST-1, fillcolor="gray", opacity=0.1,
                      annotation_text="ЗОНА ПРОГНОЗА")

        fig.update_layout(title=f"<b>Фрактальный прогноз {ticker} (LB: {LOOKBACK}d)</b>",
                          xaxis_title="Дни", yaxis_title="Цена (USD)",
                          template="plotly_white", yaxis_type="log")
        fig.show()

[*********************100%***********************]  2 of 2 completed


In [33]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- SETTINGS ---
TICKERS_COMP = ["ETH-USD", "SOL-USD"]
LB_RANGE = range(10, 181, 5)

# 1. Download Data
print(f"Downloading data for {TICKERS_COMP}...")
df_comp = yf.download(TICKERS_COMP, period="2y", interval="1d")['Close']
df_comp = df_comp.dropna()

# 2. Grid Search for Optimal Correlation
results_list = []
for lb in LB_RANGE:
    recent_segment = df_comp.iloc[-lb:]
    corr_val = recent_segment[TICKERS_COMP[0]].corr(recent_segment[TICKERS_COMP[1]])
    results_list.append({"Lookback": lb, "Correlation": corr_val})

res_df = pd.DataFrame(results_list)
best_lb_row = res_df.loc[res_df['Correlation'].idxmax()]

# 3. Visualization
fig = go.Figure()
fig.add_trace(go.Scatter(x=res_df['Lookback'], y=res_df['Correlation'],
                         mode='lines+markers', name='ETH vs SOL Correlation'))

fig.add_annotation(x=best_lb_row['Lookback'], y=best_lb_row['Correlation'],
            text=f"BEST: {int(best_lb_row['Lookback'])} days ({best_lb_row['Correlation']:.2%})",
            showarrow=True, arrowhead=1, bgcolor="#627EEA", font=dict(color="white"))

fig.update_layout(
    title="<b>Lookback Optimization: ETH vs SOL</b><br><sup>Finding the period of maximum synchronization</sup>",
    xaxis_title="Lookback Window (Days)",
    yaxis_title="Pearson Correlation",
    template="plotly_white"
)
fig.show()

print(f"Optimal window for ETH vs SOL analysis: {int(best_lb_row['Lookback'])} days.")

/tmp/ipykernel_23466/4051998511.py:12: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  2 of 2 completed

Optimal window for ETH vs SOL analysis: 160 days.


In [34]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS ---
TICKERS = ["ETH-USD", "SOL-USD"]
LOOKBACK = 160  # Optimized window from grid search
FORECAST = 60

# 1. Download full history
df_all = yf.download(TICKERS, start="2014-01-01", auto_adjust=True)['Close']
df_all = df_all.dropna()

def find_best_fractal(ticker_name, data, lb, fc):
    # Get current pattern
    current_pattern = data[ticker_name].iloc[-lb:].values
    target_norm = (current_pattern - np.mean(current_pattern)) / np.std(current_pattern)

    # Define search space (all history excluding the forecast period)
    search_space = data[ticker_name].iloc[:-fc]
    best_score = float('inf')
    best_idx = -1

    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb].values
        sample_norm = (sample - np.mean(sample)) / np.std(sample)
        score = np.mean((target_norm - sample_norm)**2)

        if score < best_score:
            best_score = score
            best_idx = i

    if best_idx != -1:
        raw_segment = data[ticker_name].iloc[best_idx : best_idx + lb + fc].values
        # Anchor to current price
        ratio = current_pattern[-1] / raw_segment[lb-1]
        projected = raw_segment * ratio
        start_date = data.index[best_idx].strftime('%Y-%m-%d')
        return projected, start_date, best_score
    return None, None, None

# 2. Execute and Visualize
for ticker in TICKERS:
    proj_vals, s_date, score = find_best_fractal(ticker, df_all, LOOKBACK, FORECAST)

    if proj_vals is not None:
        fig = go.Figure()

        # Current Price
        fig.add_trace(go.Scatter(y=df_all[ticker].iloc[-LOOKBACK:].values,
                                 name=f"Current {ticker}", line=dict(color='black', width=4)))

        # Fractal Projection
        fig.add_trace(go.Scatter(y=proj_vals,
                                 name=f"Fractal from {s_date} (MSE: {score:.3f})",
                                 line=dict(color='red', width=2, dash='dot')))

        fig.add_vrect(x0=LOOKBACK-1, x1=LOOKBACK+FORECAST-1, fillcolor="gray", opacity=0.1,
                      annotation_text="FORECAST")

        fig.update_layout(title=f"<b>Fractal Forecast: {ticker} (160d Lookback)</b>",
                          xaxis_title="Days (Relative)", yaxis_title="Price (USD)",
                          template="plotly_white", yaxis_type="log")
        fig.show()

[*********************100%***********************]  2 of 2 completed


In [35]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- CONFIGURATION ---
FORECAST = 60
PAIRS_CONFIG = [
    {"tickers": ["BTC-USD", "ETH-USD"], "lb": 175, "name": "BTC/ETH (175d)", "colors": ["#F7931A", "#627EEA"]},
    {"tickers": ["BTC-USD", "SOL-USD"], "lb": 180, "name": "BTC/SOL (180d)", "colors": ["#F7931A", "#14F195"]},
    {"tickers": ["ETH-USD", "SOL-USD"], "lb": 160, "name": "ETH/SOL (160d)", "colors": ["#627EEA", "#14F195"]}
]

# Download all unique tickers
all_tickers = list(set([t for pair in PAIRS_CONFIG for t in pair['tickers']]))
data_full = yf.download(all_tickers, start="2014-01-01", auto_adjust=True)['Close'].dropna()

def get_fractal(ticker, lb, fc):
    current = data_full[ticker].iloc[-lb:].values
    target_norm = (current - np.mean(current)) / np.std(current)
    search_space = data_full[ticker].iloc[:-fc]

    best_score = float('inf')
    best_idx = -1

    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb].values
        sample_norm = (sample - np.mean(sample)) / np.std(sample)
        score = np.mean((target_norm - sample_norm)**2)
        if score < best_score:
            best_score, best_idx = score, i

    raw = data_full[ticker].iloc[best_idx : best_idx + lb + fc].values
    # Normalize to 100 for comparison across different price scales
    proj_norm = (raw / raw[lb-1]) * 100
    return proj_norm

# Plotting
fig = go.Figure()

for config in PAIRS_CONFIG:
    lb = config['lb']
    for i, ticker in enumerate(config['tickers']):
        proj = get_fractal(ticker, lb, FORECAST)
        # Offset so all projections end at the same relative 'today'
        x_axis = np.arange(-lb + 1, FORECAST + 1)

        fig.add_trace(go.Scatter(
            x=x_axis,
            y=proj,
            name=f"{ticker} via {config['name']}",
            line=dict(color=config['colors'][i], width=2,
                      dash='solid' if i==0 else 'dot'),
            opacity=0.8
        ))

fig.add_vrect(x0=0, x1=FORECAST, fillcolor="gray", opacity=0.1, annotation_text="FORECAST")
fig.add_vline(x=0, line_width=3, line_dash="dash", line_color="black")

fig.update_layout(
    title="<b>Unified Fractal Comparison</b><br><sup>Synchronized by optimized lookback windows per asset pair</sup>",
    xaxis_title="Days from Today",
    yaxis_title="Price Relative to Today (%)",
    template="plotly_white",
    hovermode="x unified"
)
fig.show()

[*********************100%***********************]  3 of 3 completed


In [36]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS ---
TICKER = "ETH-USD"
FORECAST_WEEKS = 12
LB_RANGE_WEEKS = range(10, 53, 2) # Looking back 10 to 52 weeks

# 1. Download and Resample to Weekly
df_daily = yf.download(TICKER, start="2015-01-01", auto_adjust=True)
if isinstance(df_daily.columns, pd.MultiIndex): df_daily.columns = df_daily.columns.get_level_values(0)

df_weekly = df_daily['Close'].resample('W-MON').last().to_frame().dropna()

# 2. Grid Search for Optimal Weekly Lookback
best_weekly_corr = -1
best_weekly_lb = -1
best_weekly_idx = -1

print("Searching for optimal weekly lookback window...")

for lb in LB_RANGE_WEEKS:
    target = df_weekly['Close'].iloc[-lb:].values
    if len(target) < lb: continue

    search_space = df_weekly.iloc[:-FORECAST_WEEKS - lb]

    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        corr = np.corrcoef(target, sample)[0, 1]

        if corr > best_weekly_corr:
            best_weekly_corr = corr
            best_weekly_lb = lb
            best_weekly_idx = i

print(f"Optimal Weekly Lookback: {best_weekly_lb} weeks")
print(f"Correlation: {best_weekly_corr:.2%}")

# 3. Weekly Fractal Projection
best_sample_full = df_weekly.iloc[best_weekly_idx : best_weekly_idx + best_weekly_lb + FORECAST_WEEKS]
start_date = best_sample_full.index[0].strftime('%Y-%m-%d')

target_final = df_weekly['Close'].iloc[-best_weekly_lb:].values
ratio = target_final[-1] / best_sample_full['Close'].values[best_weekly_lb-1]
projected_weekly = best_sample_full['Close'].values * ratio

# Visualization
fig = go.Figure()

x_weeks = np.arange(-best_weekly_lb + 1, FORECAST_WEEKS + 1)

# Actual
fig.add_trace(go.Scatter(x=x_weeks[:best_weekly_lb], y=target_final,
                         name="Current Weekly ETH", line=dict(color='black', width=4)))

# Fractal
fig.add_trace(go.Scatter(x=x_weeks, y=projected_weekly,
                         name=f"Weekly Fractal from {start_date}",
                         line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST_WEEKS, fillcolor="gray", opacity=0.1, annotation_text="12-WEEK FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Weekly Fractal Forecast: {TICKER}</b><br><sup>Lookback: {best_weekly_lb} weeks | Corr: {best_weekly_corr:.2%}</sup>",
    xaxis_title="Weeks from Today",
    yaxis_title="Price (USD)",
    yaxis_type="log",
    template="plotly_white"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal weekly lookback window...
Optimal Weekly Lookback: 36 weeks
Correlation: 95.20%


In [37]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS FOR BTC WEEKLY ---
BTC_TICKER = "BTC-USD"
FORECAST_WEEKS = 12
LB_RANGE_WEEKS = range(10, 53, 2)

# 1. Download and Resample
df_daily_btc = yf.download(BTC_TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df_daily_btc.columns, pd.MultiIndex): df_daily_btc.columns = df_daily_btc.columns.get_level_values(0)

df_weekly_btc = df_daily_btc['Close'].resample('W-MON').last().to_frame().dropna()

# 2. Grid Search for Optimal Weekly Lookback
best_btc_corr = -1
best_btc_lb = -1
best_btc_idx = -1

print(f"Searching for optimal weekly lookback for {BTC_TICKER}...")

for lb in LB_RANGE_WEEKS:
    target = df_weekly_btc['Close'].iloc[-lb:].values
    if len(target) < lb: continue

    search_space = df_weekly_btc.iloc[:-FORECAST_WEEKS - lb]

    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        corr = np.corrcoef(target, sample)[0, 1]

        if corr > best_btc_corr:
            best_btc_corr = corr
            best_btc_lb = lb
            best_btc_idx = i

print(f"Optimal Weekly Lookback (BTC): {best_btc_lb} weeks")
print(f"Correlation: {best_btc_corr:.2%}")

# 3. Projection
best_sample_btc = df_weekly_btc.iloc[best_btc_idx : best_btc_idx + best_btc_lb + FORECAST_WEEKS]
s_date_btc = best_sample_btc.index[0].strftime('%Y-%m-%d')

target_final_btc = df_weekly_btc['Close'].iloc[-best_btc_lb:].values
ratio_btc = target_final_btc[-1] / best_sample_btc['Close'].values[best_btc_lb-1]
projected_btc = best_sample_btc['Close'].values * ratio_btc

# Visualization
fig = go.Figure()
x_weeks = np.arange(-best_btc_lb + 1, FORECAST_WEEKS + 1)

fig.add_trace(go.Scatter(x=x_weeks[:best_btc_lb], y=target_final_btc,
                         name="Current Weekly BTC", line=dict(color='black', width=4)))

fig.add_trace(go.Scatter(x=x_weeks, y=projected_btc,
                         name=f"Weekly Fractal from {s_date_btc}",
                         line=dict(color='#F7931A', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST_WEEKS, fillcolor="orange", opacity=0.1, annotation_text="12-WEEK BTC FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Weekly Fractal Forecast: {BTC_TICKER}</b><br><sup>Lookback: {best_btc_lb} weeks | Corr: {best_btc_corr:.2%}</sup>",
    xaxis_title="Weeks from Today",
    yaxis_title="Price (USD)",
    yaxis_type="log",
    template="plotly_white"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal weekly lookback for BTC-USD...
Optimal Weekly Lookback (BTC): 12 weeks
Correlation: 96.42%


In [38]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS FOR SOL WEEKLY ---
SOL_TICKER = "SOL-USD"
FORECAST_WEEKS = 12
LB_RANGE_WEEKS = range(10, 53, 2)

# 1. Download and Resample
df_daily_sol = yf.download(SOL_TICKER, start="2020-01-01", auto_adjust=True)
if isinstance(df_daily_sol.columns, pd.MultiIndex): df_daily_sol.columns = df_daily_sol.columns.get_level_values(0)

df_weekly_sol = df_daily_sol['Close'].resample('W-MON').last().to_frame().dropna()

# 2. Grid Search for Optimal Weekly Lookback
best_sol_corr = -1
best_sol_lb = -1
best_sol_idx = -1

print(f"Searching for optimal weekly lookback for {SOL_TICKER}...")

for lb in LB_RANGE_WEEKS:
    target = df_weekly_sol['Close'].iloc[-lb:].values
    if len(target) < lb: continue

    search_space = df_weekly_sol.iloc[:-FORECAST_WEEKS - lb]

    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        corr = np.corrcoef(target, sample)[0, 1]

        if corr > best_sol_corr:
            best_sol_corr = corr
            best_sol_lb = lb
            best_sol_idx = i

print(f"Optimal Weekly Lookback (SOL): {best_sol_lb} weeks")
print(f"Correlation: {best_sol_corr:.2%}")

# 3. Projection
best_sample_sol = df_weekly_sol.iloc[best_sol_idx : best_sol_idx + best_sol_lb + FORECAST_WEEKS]
s_date_sol = best_sample_sol.index[0].strftime('%Y-%m-%d')

target_final_sol = df_weekly_sol['Close'].iloc[-best_sol_lb:].values
ratio_sol = target_final_sol[-1] / best_sample_sol['Close'].values[best_sol_lb-1]
projected_sol = best_sample_sol['Close'].values * ratio_sol

# Visualization
fig = go.Figure()
x_weeks = np.arange(-best_sol_lb + 1, FORECAST_WEEKS + 1)

fig.add_trace(go.Scatter(x=x_weeks[:best_sol_lb], y=target_final_sol,
                         name="Current Weekly SOL", line=dict(color='black', width=4)))

fig.add_trace(go.Scatter(x=x_weeks, y=projected_sol,
                         name=f"Weekly Fractal from {s_date_sol}",
                         line=dict(color='#14F195', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST_WEEKS, fillcolor="#14F195", opacity=0.1, annotation_text="12-WEEK SOL FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Weekly Fractal Forecast: {SOL_TICKER}</b><br><sup>Lookback: {best_sol_lb} weeks | Corr: {best_sol_corr:.2%}</sup>",
    xaxis_title="Weeks from Today",
    yaxis_title="Price (USD)",
    yaxis_type="log",
    template="plotly_white"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal weekly lookback for SOL-USD...
Optimal Weekly Lookback (SOL): 36 weeks
Correlation: 93.73%


In [39]:
import plotly.graph_objects as go
import numpy as np

# --- CONSOLIDATED WEEKLY COMPARISON ---
# Redefining projections to ensure variables are available
FORECAST_WEEKS = 12
fig_comp = go.Figure()

# Helper to get the normalized projection for the plot
def get_norm_proj(proj_vals, lb_len):
    # Index of 'today' is lb_len - 1
    return (proj_vals / proj_vals[lb_len-1]) * 100

try:
    # 1. BTC Projection
    btc_proj_norm = get_norm_proj(projected_btc, best_btc_lb)
    fig_comp.add_trace(go.Scatter(
        x=np.arange(-best_btc_lb + 1, FORECAST_WEEKS + 1),
        y=btc_proj_norm,
        name=f"BTC (Match: {best_btc_corr:.1%})",
        line=dict(color='#F7931A', width=3)
    ))

    # 2. ETH Projection (Variable was named projected_weekly in its cell)
    eth_proj_norm = get_norm_proj(projected_weekly, best_weekly_lb)
    fig_comp.add_trace(go.Scatter(
        x=np.arange(-best_weekly_lb + 1, FORECAST_WEEKS + 1),
        y=eth_proj_norm,
        name=f"ETH (Match: {best_weekly_corr:.1%})",
        line=dict(color='#627EEA', width=3)
    ))

    # 3. SOL Projection
    sol_proj_norm = get_norm_proj(projected_sol, best_sol_lb)
    fig_comp.add_trace(go.Scatter(
        x=np.arange(-best_sol_lb + 1, FORECAST_WEEKS + 1),
        y=sol_proj_norm,
        name=f"SOL (Match: {best_sol_corr:.1%})",
        line=dict(color='#14F195', width=3)
    ))

    # Formatting
    fig_comp.add_vrect(x0=0, x1=FORECAST_WEEKS, fillcolor="gray", opacity=0.1, annotation_text="FORECAST")
    fig_comp.add_vline(x=0, line_width=2, line_dash="dash", line_color="black")

    fig_comp.update_layout(
        title="<b>Weekly Market Synthesis: BTC vs ETH vs SOL</b>",
        xaxis_title="Weeks from Today",
        yaxis_title="Relative Performance (%)",
        template="plotly_white",
        hovermode="x unified"
    )
    fig_comp.show()
except NameError as e:
    print(f"Error: {e}. Please ensure the individual BTC, ETH, and SOL weekly cells have been executed first.")

### Correlation Analysis: Crypto vs. US Treasury Yields
This section investigates the relationship between the crypto market and bond yields (^IRX, ^FVX, ^TNX, ^TYX). We will perform a grid search to find the 'Lookback' window where the correlation is most significant.

In [40]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- SETTINGS ---
CRYPTO_TICKERS = ["BTC-USD", "ETH-USD", "SOL-USD"]
YIELD_TICKERS = {
    "^IRX": "13-Week Bill",
    "^FVX": "5-Year Yield",
    "^TNX": "10-Year Yield",
    "^TYX": "30-Year Yield"
}
LB_RANGE = range(10, 251, 10) # Testing windows from 10 to 250 days

# 1. Download Data
print("Downloading Treasury and Crypto data...")
all_req_tickers = CRYPTO_TICKERS + list(YIELD_TICKERS.keys())
data_corr = yf.download(all_req_tickers, period="3y", interval="1d", auto_adjust=True)['Close']
data_corr = data_corr.fillna(method='ffill').dropna()

# 2. Grid Search for Best Correlation Window
# We will focus on ^TNX (10Y) as the primary benchmark for this search
benchmark_yield = "^TNX"
search_results = []

for crypto in CRYPTO_TICKERS:
    for lb in LB_RANGE:
        segment = data_corr.iloc[-lb:]
        corr_val = segment[crypto].corr(segment[benchmark_yield])
        search_results.append({"Crypto": crypto, "Lookback": lb, "Correlation": corr_val})

res_df = pd.DataFrame(search_results)

# 3. Visualization: Correlation Heatmap/Grid
fig = go.Figure()
for crypto in CRYPTO_TICKERS:
    crypto_res = res_df[res_df['Crypto'] == crypto]
    fig.add_trace(go.Scatter(x=crypto_res['Lookback'], y=crypto_res['Correlation'],
                             mode='lines+markers', name=f"{crypto} vs {benchmark_yield}"))

fig.update_layout(
    title=f"<b>Lookback Optimization: Crypto vs {YIELD_TICKERS[benchmark_yield]}</b>",
    xaxis_title="Lookback Window (Days)",
    yaxis_title="Pearson Correlation",
    template="plotly_white",
    hovermode="x unified"
)
fig.show()

# Summary table for best windows
print("\nOptimal Correlation Windows (Benchmark: 10Y Yield):")
for crypto in CRYPTO_TICKERS:
    best_row = res_df[res_df['Crypto'] == crypto].loc[res_df[res_df['Crypto'] == crypto]['Correlation'].abs().idxmax()]
    print(f"{crypto}: Best LB = {int(best_row['Lookback'])} days | Corr = {best_row['Correlation']:.2%}")

[*********************100%***********************]  7 of 7 completed
/tmp/ipykernel_23466/421904415.py:21: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.




Optimal Correlation Windows (Benchmark: 10Y Yield):
BTC-USD: Best LB = 30 days | Corr = 56.67%
ETH-USD: Best LB = 90 days | Corr = 60.84%
SOL-USD: Best LB = 10 days | Corr = -61.74%


In [41]:
# --- MULTI-YIELD COMPARISON CHART ---
# Visualizing all yields vs BTC for the last 180 days
LB_VISUAL = 180
recent_data = data_corr.iloc[-LB_VISUAL:]

fig_multi = make_subplots(specs=[[{"secondary_y": True}]])

# Crypto (Primary Y)
fig_multi.add_trace(go.Scatter(x=recent_data.index, y=recent_data["BTC-USD"],
                               name="BTC-USD", line=dict(color='black', width=3)), secondary_y=False)

# Yields (Secondary Y)
colors_y = ['red', 'blue', 'green', 'orange']
for i, (ytick, yname) in enumerate(YIELD_TICKERS.items()):
    fig_multi.add_trace(go.Scatter(x=recent_data.index, y=recent_data[ytick],
                                   name=yname, line=dict(color=colors_y[i], dash='dot')), secondary_y=True)

fig_multi.update_layout(title=f"<b>Market Context: BTC vs US Treasury Yields (Last {LB_VISUAL}d)</b>",
                        template="plotly_white", hovermode="x unified")
fig_multi.update_yaxes(title_text="BTC Price", secondary_y=False)
fig_multi.update_yaxes(title_text="Yield %", secondary_y=True)
fig_multi.show()

In [43]:
# --- YIELD SENSITIVITY MATRIX ---
import plotly.express as px

full_res = []
for crypto in CRYPTO_TICKERS:
    for y_tick, y_name in YIELD_TICKERS.items():
        for lb in LB_RANGE:
            segment = data_corr.iloc[-lb:]
            c = segment[crypto].corr(segment[y_tick])
            full_res.append({'Crypto': crypto, 'Yield': y_name, 'LB': lb, 'Corr': c})

sensitivity_df = pd.DataFrame(full_res)

print("Dominant Macro Indicators per Asset:")
for crypto in CRYPTO_TICKERS:
    subset = sensitivity_df[sensitivity_df['Crypto'] == crypto]
    best_idx = subset['Corr'].abs().idxmax()
    best = subset.loc[best_idx]
    print(f"{crypto}: Most sensitive to {best['Yield']} ({best['LB']}d window) | Corr: {best['Corr']:.2%}")

# Visualization: 90-day correlation heatmap
corr_matrix = data_corr.iloc[-90:].corr()
filtered_corr = corr_matrix.loc[CRYPTO_TICKERS, list(YIELD_TICKERS.keys())]
filtered_corr.columns = [YIELD_TICKERS[c] for c in filtered_corr.columns]

fig_heat = px.imshow(filtered_corr, text_auto='.2f', color_continuous_scale='RdBu_r', title='Crypto vs Treasury Yields (90-Day Correlation Heatmap)')
fig_heat.show()

Dominant Macro Indicators per Asset:
BTC-USD: Most sensitive to 13-Week Bill (10d window) | Corr: 82.99%
ETH-USD: Most sensitive to 13-Week Bill (150d window) | Corr: -78.42%
SOL-USD: Most sensitive to 13-Week Bill (250d window) | Corr: 82.69%


### Final Synthesis: Macro-Aware 12-Week Synthesis
This section merges the high-confidence fractal projections from Phase 1 with the yield sensitivity data from Phase 2. We present a unified relative performance chart for the next 12 weeks.

In [45]:
import numpy as np
import plotly.graph_objects as go

# --- FINAL MARKET SYNTHESIS ---
fig_final = go.Figure()

def get_norm_proj(proj_vals, lb_len):
    return (proj_vals / proj_vals[lb_len-1]) * 100

# Data consolidation from previous phases
synthesis_data = [
    {'ticker': 'BTC', 'proj': projected_btc, 'lb': best_btc_lb, 'corr': 0.83, 'color': '#F7931A'},
    {'ticker': 'ETH', 'proj': projected_weekly, 'lb': best_weekly_lb, 'corr': -0.78, 'color': '#627EEA'},
    {'ticker': 'SOL', 'proj': projected_sol, 'lb': best_sol_lb, 'corr': 0.83, 'color': '#14F195'}
]

for asset in synthesis_data:
    norm_vals = get_norm_proj(asset['proj'], asset['lb'])
    x_range = np.arange(-asset['lb'] + 1, FORECAST_WEEKS + 1)

    fig_final.add_trace(go.Scatter(
        x=x_range,
        y=norm_vals,
        name=f"{asset['ticker']} (Yield Corr: {asset['corr']})",
        line=dict(color=asset['color'], width=4)
    ))

# Visual markers
fig_final.add_vrect(x0=0, x1=FORECAST_WEEKS, fillcolor="gray", opacity=0.1, annotation_text="MACRO-AWARE FORECAST")
fig_final.add_vline(x=0, line_width=2, line_dash="dash", line_color="black")

fig_final.update_layout(
    title="<b>Unified 12-Week Forecast Synthesis</b><br><sup>Combining Weekly Fractals with 13-Week Treasury Bill Sensitivity</sup>",
    xaxis_title="Weeks from Today",
    yaxis_title="Relative Performance (% of Today)",
    template="plotly_white",
    hovermode="x unified"
)
fig_final.show()

In [46]:
# --- ETH VS LONG-TERM YIELDS COMPARISON ---
# Using the 180-day window for visual clarity on long-term trends
LB_VISUAL_LONG = 180
recent_data_eth = data_corr.iloc[-LB_VISUAL_LONG:]

fig_eth_yields = make_subplots(specs=[[{"secondary_y": True}]])

# Ethereum (Primary Y)
fig_eth_yields.add_trace(go.Scatter(x=recent_data_eth.index, y=recent_data_eth["ETH-USD"],
                                    name="ETH-USD", line=dict(color='#627EEA', width=4)), secondary_y=False)

# Long-term Yields (Secondary Y)
long_yields = {"^FVX": "5-Year Yield", "^TNX": "10-Year Yield", "^TYX": "30-Year Yield"}
yield_colors = {'^FVX': '#FF6B6B', '^TNX': '#4ECDC4', '^TYX': '#45B7D1'}

for ytick, yname in long_yields.items():
    fig_eth_yields.add_trace(go.Scatter(x=recent_data_eth.index, y=recent_data_eth[ytick],
                                       name=yname, line=dict(color=yield_colors[ytick], dash='dot')), secondary_y=True)

fig_eth_yields.update_layout(title=f"<b>ETH vs Long-Term Treasury Yields (Last {LB_VISUAL_LONG}d)</b>",
                             template="plotly_white", hovermode="x unified")
fig_eth_yields.update_yaxes(title_text="ETH Price (USD)", secondary_y=False)
fig_eth_yields.update_yaxes(title_text="Yield %", secondary_y=True)
fig_eth_yields.show()

In [47]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- LOOKBACK OPTIMIZATION FOR ETH VS BONDS ---
ETH_TICKER = "ETH-USD"
BOND_BENCHMARK = "^TNX"  # 10-Year Yield
LB_SEARCH_RANGE = range(10, 251, 5)

# Using existing data_corr from previous cells
eth_bond_results = []

for lb in LB_SEARCH_RANGE:
    segment = data_corr.iloc[-lb:]
    corr_val = segment[ETH_TICKER].corr(segment[BOND_BENCHMARK])
    eth_bond_results.append({"Lookback": lb, "Correlation": corr_val})

eth_bond_df = pd.DataFrame(eth_bond_results)
best_lb_eth_bond = eth_bond_df.loc[eth_bond_df['Correlation'].abs().idxmax()]

# Visualization of the search
fig_opt = go.Figure()
fig_opt.add_trace(go.Scatter(x=eth_bond_df['Lookback'], y=eth_bond_df['Correlation'],
                             mode='lines+markers', name='ETH vs 10Y Yield Correlation'))

fig_opt.add_annotation(x=best_lb_eth_bond['Lookback'], y=best_lb_eth_bond['Correlation'],
            text=f"BEST: {int(best_lb_eth_bond['Lookback'])} days ({best_lb_eth_bond['Correlation']:.2%})",
            showarrow=True, arrowhead=1, bgcolor="#627EEA", font=dict(color="white"))

fig_opt.update_layout(
    title=f"<b>Lookback Optimization: {ETH_TICKER} vs {BOND_BENCHMARK}</b>",
    xaxis_title="Lookback Window (Days)",
    yaxis_title="Pearson Correlation",
    template="plotly_white"
)
fig_opt.show()

print(f"Optimal window for ETH vs Bond analysis: {int(best_lb_eth_bond['Lookback'])} days.")

Optimal window for ETH vs Bond analysis: 95 days.


In [57]:
import numpy as np
import plotly.graph_objects as go

# --- UNIFIED BTC & ETH FORECAST ---
fig_unified = go.Figure()

# 1. BTC Data (from cell 75273d64 variables)
btc_x = np.arange(-best_btc_overall_lb + 1, FORECAST_DAYS + 1)
btc_norm = (projected_vals_btc / projected_vals_btc[best_btc_overall_lb-1]) * 100

fig_unified.add_trace(go.Scatter(
    x=btc_x,
    y=btc_norm,
    name=f"BTC Fractal (Match: {best_btc_overall_corr:.1%})",
    line=dict(color='#F7931A', width=3)
))

# 2. ETH Data (from cell 293ad385 variables)
eth_x = np.arange(-best_overall_lb + 1, FORECAST + 1)
eth_norm = (projected_vals / projected_vals[best_overall_lb-1]) * 100

fig_unified.add_trace(go.Scatter(
    x=eth_x,
    y=eth_norm,
    name=f"ETH Fractal (Match: {best_overall_corr:.1%})",
    line=dict(color='#627EEA', width=3)
))

# Formatting
fig_unified.add_vrect(x0=0, x1=60, fillcolor="rgba(128,128,128,0.1)", annotation_text="60-DAY CONFLUENCE ZONE")
fig_unified.add_vline(x=0, line_width=2, line_dash="dash", line_color="black")

fig_unified.update_layout(
    title="<b>Unified Market Forecast: BTC vs ETH Fractal Confluence</b>",
    xaxis_title="Days from Today",
    yaxis_title="Relative Performance (% of Current Price)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig_unified.show()

In [56]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS ---
TICKER_BTC = "BTC-USD"
FORECAST_DAYS = 60
LB_MIN = 30  # REFINED: Minimum 30 days to avoid 2-day noise
LB_MAX = 365

# 1. Download Data
df_btc_full = yf.download(TICKER_BTC, start="2014-01-01", auto_adjust=True)
if isinstance(df_btc_full.columns, pd.MultiIndex): df_btc_full.columns = df_btc_full.columns.get_level_values(0)
df_btc_full = df_btc_full['Close'].to_frame().dropna()

# 2. Grid Search for Best Lookback (30 to 365 days)
best_btc_overall_corr = -1
best_btc_overall_lb = -1
best_btc_overall_idx = -1

print(f"Searching for optimal BTC cycle window ({LB_MIN}-{LB_MAX} days)...")

for lb in range(LB_MIN, LB_MAX + 1):
    if len(df_btc_full) < lb + FORECAST_DAYS: continue

    current_pattern = df_btc_full['Close'].iloc[-lb:].values
    if len(current_pattern) > 1 and np.std(current_pattern) > 0:
        search_space = df_btc_full.iloc[:-FORECAST_DAYS - lb]

        for i in range(len(search_space) - lb):
            sample = search_space.iloc[i : i + lb]['Close'].values
            if np.std(sample) > 0:
                corr = np.corrcoef(current_pattern, sample)[0, 1]

                if corr > best_btc_overall_corr:
                    best_btc_overall_corr = corr
                    best_btc_overall_lb = lb
                    best_btc_overall_idx = i

print(f"Optimal BTC Cycle Found: {best_btc_overall_lb} days")
print(f"Correlation Strength: {best_btc_overall_corr:.2%}")

# 3. Extract and Project
best_sample_btc = df_btc_full.iloc[best_btc_overall_idx : best_btc_overall_idx + best_btc_overall_lb + FORECAST_DAYS]
start_date_btc = best_sample_btc.index[0].strftime('%Y-%m-%d')

target_final_btc = df_btc_full['Close'].iloc[-best_btc_overall_lb:].values
ratio_btc = target_final_btc[-1] / best_sample_btc['Close'].values[best_btc_overall_lb-1]
projected_vals_btc = best_sample_btc['Close'].values * ratio_btc

# 4. Visualization
fig_btc = go.Figure()
x_axis_btc = np.arange(-best_btc_overall_lb + 1, FORECAST_DAYS + 1)

fig_btc.add_trace(go.Scatter(x=x_axis_btc[:best_btc_overall_lb], y=target_final_btc,
                         name="Current BTC Price", line=dict(color='black', width=4)))

fig_btc.add_trace(go.Scatter(x=x_axis_btc, y=projected_vals_btc,
                         name=f"Best BTC Fractal ({start_date_btc})",
                         line=dict(color='#F7931A', width=2, dash='dot')))

fig_btc.add_vrect(x0=0, x1=FORECAST_DAYS, fillcolor="orange", opacity=0.05, annotation_text="FORECAST")
fig_btc.add_vline(x=0, line_dash="dash", line_color="black")

fig_btc.update_layout(
    title=f"<b>Refined BTC Fractal Optimization ({LB_MIN}-{LB_MAX}d)</b><br><sup>Optimal Window: {best_btc_overall_lb} days | Correlation: {best_btc_overall_corr:.2%}</sup>",
    xaxis_title="Days from Today",
    yaxis_title="Price (USD)",
    yaxis_type="log",
    template="plotly_white",
    hovermode="x unified"
)
fig_btc.show()

print(f"Best BTC historical match starts: {start_date_btc}")

[*********************100%***********************]  1 of 1 completed


Searching for optimal BTC cycle window (30-365 days)...
Optimal BTC Cycle Found: 46 days
Correlation Strength: 96.16%


Best BTC historical match starts: 2025-04-12


In [53]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS ---
TICKER = "ETH-USD"
FORECAST = 60
LB_MIN = 30  # Increased minimum to avoid noise/2-day artifacts
LB_MAX = 365

# 1. Download Data
df = yf.download(TICKER, start="2015-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
df = df['Close'].to_frame().dropna()

# 2. Grid Search for Absolute Best Lookback (30 to 365 days)
best_overall_corr = -1
best_overall_lb = -1
best_overall_idx = -1

print(f"Searching for optimal cycle window ({LB_MIN}-{LB_MAX} days)...")

for lb in range(LB_MIN, LB_MAX + 1):
    if len(df) < lb + FORECAST: continue

    current_pattern = df['Close'].iloc[-lb:].values
    if len(current_pattern) > 1 and np.std(current_pattern) > 0:
        search_space = df.iloc[:-FORECAST - lb]

        for i in range(len(search_space) - lb):
            sample = search_space.iloc[i : i + lb]['Close'].values
            if np.std(sample) > 0:
                corr = np.corrcoef(current_pattern, sample)[0, 1]

                if corr > best_overall_corr:
                    best_overall_corr = corr
                    best_overall_lb = lb
                    best_overall_idx = i

print(f"Optimal Cycle Found: {best_overall_lb} days")
print(f"Correlation Strength: {best_overall_corr:.2%}")

# 3. Extract and Project
best_sample_full = df.iloc[best_overall_idx : best_overall_idx + best_overall_lb + FORECAST]
start_date = best_sample_full.index[0].strftime('%Y-%m-%d')

target_final = df['Close'].iloc[-best_overall_lb:].values
ratio = target_final[-1] / best_sample_full['Close'].values[best_overall_lb-1]
projected_vals = best_sample_full['Close'].values * ratio

# 4. Visualization
fig = go.Figure()
x_axis = np.arange(-best_overall_lb + 1, FORECAST + 1)

# Actual
fig.add_trace(go.Scatter(x=x_axis[:best_overall_lb], y=target_final,
                         name="Current Price", line=dict(color='black', width=4)))

# Fractal
fig.add_trace(go.Scatter(x=x_axis, y=projected_vals,
                         name=f"Best Fractal ({start_date})",
                         line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST, fillcolor="gray", opacity=0.1, annotation_text="FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Refined Global Optimization ({LB_MIN}-{LB_MAX}d): {TICKER}</b><br><sup>Cycle Match: {best_overall_lb} days | Correlation: {best_overall_corr:.2%}</sup>",
    xaxis_title="Days from Today",
    yaxis_title="Price (USD)",
    yaxis_type="log",
    template="plotly_white",
    hovermode="x unified"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal cycle window (30-365 days)...
Optimal Cycle Found: 263 days
Correlation Strength: 95.72%


In [61]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS ---
TICKER_SOL = "SOL-USD"
FORECAST = 60
LB_MIN = 30
LB_MAX = 365

# 1. Download Data
df_sol = yf.download(TICKER_SOL, start="2020-01-01", auto_adjust=True)
if isinstance(df_sol.columns, pd.MultiIndex): df_sol.columns = df_sol.columns.get_level_values(0)
df_sol = df_sol['Close'].to_frame().dropna()

# 2. Grid Search for Absolute Best Lookback (30 to 365 days)
best_sol_overall_corr = -1
best_sol_overall_lb = -1
best_sol_overall_idx = -1

print(f"Searching for optimal SOL cycle window ({LB_MIN}-{LB_MAX} days)...")

for lb in range(LB_MIN, LB_MAX + 1):
    if len(df_sol) < lb + FORECAST: continue

    current_pattern = df_sol['Close'].iloc[-lb:].values
    if len(current_pattern) > 1 and np.std(current_pattern) > 0:
        search_space = df_sol.iloc[:-FORECAST - lb]

        for i in range(len(search_space) - lb):
            sample = search_space.iloc[i : i + lb]['Close'].values
            if np.std(sample) > 0:
                corr = np.corrcoef(current_pattern, sample)[0, 1]

                if corr > best_sol_overall_corr:
                    best_sol_overall_corr = corr
                    best_sol_overall_lb = lb
                    best_sol_overall_idx = i

print(f"Optimal SOL Cycle Found: {best_sol_overall_lb} days")
print(f"Correlation Strength: {best_sol_overall_corr:.2%}")

# 3. Extract and Project
best_sample_sol = df_sol.iloc[best_sol_overall_idx : best_sol_overall_idx + best_sol_overall_lb + FORECAST]
start_date_sol = best_sample_sol.index[0].strftime('%Y-%m-%d')

target_final_sol = df_sol['Close'].iloc[-best_sol_overall_lb:].values
ratio_sol = target_final_sol[-1] / best_sample_sol['Close'].values[best_sol_overall_lb-1]
projected_vals_sol = best_sample_sol['Close'].values * ratio_sol

# 4. Visualization
fig_sol = go.Figure()
x_axis_sol = np.arange(-best_sol_overall_lb + 1, FORECAST + 1)

# Actual
fig_sol.add_trace(go.Scatter(x=x_axis_sol[:best_sol_overall_lb], y=target_final_sol,
                         name="Current SOL Price", line=dict(color='black', width=4)))

# Fractal
fig_sol.add_trace(go.Scatter(x=x_axis_sol, y=projected_vals_sol,
                         name=f"Best SOL Fractal ({start_date_sol})",
                         line=dict(color='#14F195', width=2, dash='dot')))

fig_sol.add_vrect(x0=0, x1=FORECAST, fillcolor="#14F195", opacity=0.05, annotation_text="FORECAST")
fig_sol.add_vline(x=0, line_dash="dash", line_color="black")

fig_sol.update_layout(
    title=f"<b>Refined SOL Global Optimization ({LB_MIN}-{LB_MAX}d)</b><br><sup>Cycle Match: {best_sol_overall_lb} days | Correlation: {best_sol_overall_corr:.2%}</sup>",
    xaxis_title="Days from Today",
    yaxis_title="Price (USD)",
    yaxis_type="log",
    template="plotly_white",
    hovermode="x unified"
)
fig_sol.show()

print(f"Best SOL historical match starts: {start_date_sol}")

[*********************100%***********************]  1 of 1 completed


Searching for optimal SOL cycle window (30-365 days)...
Optimal SOL Cycle Found: 129 days
Correlation Strength: 95.51%


Best SOL historical match starts: 2022-04-10


In [69]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- EXPANDED SETTINGS FOR LARGE-SCALE INTRADAY OPTIMIZATION ---
TICKER = "ETH-USD"
# Expanded range to 1-1440 hours (up to 60 days)
LOOKBACK_RANGE = range(1, 24)
FORECAST_HOURS = 12

# 1. Download Hourly Data
print(f"Downloading hourly data for {TICKER}...")
# 730 days is the maximum allowed for 1h interval via yfinance
df_hourly = yf.download(TICKER, period="730d", interval="1h", auto_adjust=True)
if isinstance(df_hourly.columns, pd.MultiIndex): df_hourly.columns = df_hourly.columns.get_level_values(0)
df_hourly = df_hourly['Close'].to_frame().dropna()

# 2. Grid Search for Optimal Hourly Lookback
hourly_results = []

print(f"Searching for optimal intraday cycle window (1-24h)... ")

for lb in LOOKBACK_RANGE:
    current_pattern = df_hourly['Close'].iloc[-lb:].values
    if len(current_pattern) < lb or np.std(current_pattern) == 0: continue

    # Define search space excluding the forecast and current window
    search_space = df_hourly.iloc[:-FORECAST_HOURS - lb]
    best_corr = -1
    best_idx = -1

    # For large ranges, we skip to find best match efficiently
    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        if np.std(sample) > 0:
            corr = np.corrcoef(current_pattern, sample)[0, 1]
            if corr > best_corr:
                best_corr = corr
                best_idx = i

    hourly_results.append({"Lookback_Hours": lb, "Correlation": best_corr, "Index": best_idx})

res_df = pd.DataFrame(hourly_results)
best_lb_row = res_df.loc[res_df['Correlation'].idxmax()]
opt_lb = int(best_lb_row['Lookback_Hours'])
opt_corr = best_lb_row['Correlation']
opt_idx = int(best_lb_row['Index'])

print(f"\nOptimal Window: {opt_lb} hours")
print(f"Max Correlation: {opt_corr:.2%}")

# 3. Visualization
best_sample = df_hourly.iloc[opt_idx : opt_idx + opt_lb + FORECAST_HOURS]
start_time = best_sample.index[0].strftime('%Y-%m-%d %H:%M')

target_final = df_hourly['Close'].iloc[-opt_lb:].values
ratio = target_final[-1] / best_sample['Close'].values[opt_lb-1]
projected_vals = best_sample['Close'].values * ratio

fig = go.Figure()
x_axis = np.arange(-opt_lb + 1, FORECAST_HOURS + 1)

fig.add_trace(go.Scatter(x=x_axis[:opt_lb], y=target_final, name="Current (Hourly)", line=dict(color='black', width=4)))
fig.add_trace(go.Scatter(x=x_axis, y=projected_vals, name=f"Fractal (Start: {start_time})", line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST_HOURS, fillcolor="rgba(255,0,0,0.05)", annotation_text="12H FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Global Hourly Fractal Optimization (1-1440h Search): {TICKER}</b><br><sup>Best Window: {opt_lb} Hours | Correlation: {opt_corr:.2%}</sup>",
    xaxis_title="Hours from Now", yaxis_title="Price (USD)", template="plotly_white"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal intraday cycle window (1-1440h)... 

Optimal Window: 2 hours
Max Correlation: 100.00%


In [70]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- EXPANDED SETTINGS FOR LARGE-SCALE INTRADAY OPTIMIZATION ---
TICKER = "EURUSD=X"
# Expanded range to 1-1440 hours (up to 60 days)
LOOKBACK_RANGE = range(1, 24)
FORECAST_HOURS = 12

# 1. Download Hourly Data
print(f"Downloading hourly data for {TICKER}...")
# 730 days is the maximum allowed for 1h interval via yfinance
df_hourly = yf.download(TICKER, period="730d", interval="1h", auto_adjust=True)
if isinstance(df_hourly.columns, pd.MultiIndex): df_hourly.columns = df_hourly.columns.get_level_values(0)
df_hourly = df_hourly['Close'].to_frame().dropna()

# 2. Grid Search for Optimal Hourly Lookback
hourly_results = []

print(f"Searching for optimal intraday cycle window (1-24h)... ")

for lb in LOOKBACK_RANGE:
    current_pattern = df_hourly['Close'].iloc[-lb:].values
    if len(current_pattern) < lb or np.std(current_pattern) == 0: continue

    # Define search space excluding the forecast and current window
    search_space = df_hourly.iloc[:-FORECAST_HOURS - lb]
    best_corr = -1
    best_idx = -1

    # For large ranges, we skip to find best match efficiently
    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        if np.std(sample) > 0:
            corr = np.corrcoef(current_pattern, sample)[0, 1]
            if corr > best_corr:
                best_corr = corr
                best_idx = i

    hourly_results.append({"Lookback_Hours": lb, "Correlation": best_corr, "Index": best_idx})

res_df = pd.DataFrame(hourly_results)
best_lb_row = res_df.loc[res_df['Correlation'].idxmax()]
opt_lb = int(best_lb_row['Lookback_Hours'])
opt_corr = best_lb_row['Correlation']
opt_idx = int(best_lb_row['Index'])

print(f"\nOptimal Window: {opt_lb} hours")
print(f"Max Correlation: {opt_corr:.2%}")

# 3. Visualization
best_sample = df_hourly.iloc[opt_idx : opt_idx + opt_lb + FORECAST_HOURS]
start_time = best_sample.index[0].strftime('%Y-%m-%d %H:%M')

target_final = df_hourly['Close'].iloc[-opt_lb:].values
ratio = target_final[-1] / best_sample['Close'].values[opt_lb-1]
projected_vals = best_sample['Close'].values * ratio

fig = go.Figure()
x_axis = np.arange(-opt_lb + 1, FORECAST_HOURS + 1)

fig.add_trace(go.Scatter(x=x_axis[:opt_lb], y=target_final, name="Current (Hourly)", line=dict(color='black', width=4)))
fig.add_trace(go.Scatter(x=x_axis, y=projected_vals, name=f"Fractal (Start: {start_time})", line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST_HOURS, fillcolor="rgba(255,0,0,0.05)", annotation_text="12H FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Global Hourly Fractal Optimization (1-24h Search): {TICKER}</b><br><sup>Best Window: {opt_lb} Hours | Correlation: {opt_corr:.2%}</sup>",
    xaxis_title="Hours from Now", yaxis_title="Price (USD)", template="plotly_white"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal intraday cycle window (1-24h)... 

Optimal Window: 3 hours
Max Correlation: 100.00%


In [71]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- EXPANDED SETTINGS FOR LARGE-SCALE INTRADAY OPTIMIZATION ---
TICKER = "USDCHF=X"
# Expanded range to 1-1440 hours (up to 60 days)
LOOKBACK_RANGE = range(1, 24)
FORECAST_HOURS = 12

# 1. Download Hourly Data
print(f"Downloading hourly data for {TICKER}...")
# 730 days is the maximum allowed for 1h interval via yfinance
df_hourly = yf.download(TICKER, period="730d", interval="1h", auto_adjust=True)
if isinstance(df_hourly.columns, pd.MultiIndex): df_hourly.columns = df_hourly.columns.get_level_values(0)
df_hourly = df_hourly['Close'].to_frame().dropna()

# 2. Grid Search for Optimal Hourly Lookback
hourly_results = []

print(f"Searching for optimal intraday cycle window (1-24h)... ")

for lb in LOOKBACK_RANGE:
    current_pattern = df_hourly['Close'].iloc[-lb:].values
    if len(current_pattern) < lb or np.std(current_pattern) == 0: continue

    # Define search space excluding the forecast and current window
    search_space = df_hourly.iloc[:-FORECAST_HOURS - lb]
    best_corr = -1
    best_idx = -1

    # For large ranges, we skip to find best match efficiently
    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        if np.std(sample) > 0:
            corr = np.corrcoef(current_pattern, sample)[0, 1]
            if corr > best_corr:
                best_corr = corr
                best_idx = i

    hourly_results.append({"Lookback_Hours": lb, "Correlation": best_corr, "Index": best_idx})

res_df = pd.DataFrame(hourly_results)
best_lb_row = res_df.loc[res_df['Correlation'].idxmax()]
opt_lb = int(best_lb_row['Lookback_Hours'])
opt_corr = best_lb_row['Correlation']
opt_idx = int(best_lb_row['Index'])

print(f"\nOptimal Window: {opt_lb} hours")
print(f"Max Correlation: {opt_corr:.2%}")

# 3. Visualization
best_sample = df_hourly.iloc[opt_idx : opt_idx + opt_lb + FORECAST_HOURS]
start_time = best_sample.index[0].strftime('%Y-%m-%d %H:%M')

target_final = df_hourly['Close'].iloc[-opt_lb:].values
ratio = target_final[-1] / best_sample['Close'].values[opt_lb-1]
projected_vals = best_sample['Close'].values * ratio

fig = go.Figure()
x_axis = np.arange(-opt_lb + 1, FORECAST_HOURS + 1)

fig.add_trace(go.Scatter(x=x_axis[:opt_lb], y=target_final, name="Current (Hourly)", line=dict(color='black', width=4)))
fig.add_trace(go.Scatter(x=x_axis, y=projected_vals, name=f"Fractal (Start: {start_time})", line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST_HOURS, fillcolor="rgba(255,0,0,0.05)", annotation_text="12H FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Global Hourly Fractal Optimization (1-24h Search): {TICKER}</b><br><sup>Best Window: {opt_lb} Hours | Correlation: {opt_corr:.2%}</sup>",
    xaxis_title="Hours from Now", yaxis_title="Price (USD)", template="plotly_white"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal intraday cycle window (1-24h)... 

Optimal Window: 2 hours
Max Correlation: 100.00%


In [72]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- EXPANDED SETTINGS FOR LARGE-SCALE INTRADAY OPTIMIZATION ---
TICKER = "DX-Y.NYB"
# Expanded range to 1-1440 hours (up to 60 days)
LOOKBACK_RANGE = range(1, 24)
FORECAST_HOURS = 12

# 1. Download Hourly Data
print(f"Downloading hourly data for {TICKER}...")
# 730 days is the maximum allowed for 1h interval via yfinance
df_hourly = yf.download(TICKER, period="730d", interval="1h", auto_adjust=True)
if isinstance(df_hourly.columns, pd.MultiIndex): df_hourly.columns = df_hourly.columns.get_level_values(0)
df_hourly = df_hourly['Close'].to_frame().dropna()

# 2. Grid Search for Optimal Hourly Lookback
hourly_results = []

print(f"Searching for optimal intraday cycle window (1-24h)... ")

for lb in LOOKBACK_RANGE:
    current_pattern = df_hourly['Close'].iloc[-lb:].values
    if len(current_pattern) < lb or np.std(current_pattern) == 0: continue

    # Define search space excluding the forecast and current window
    search_space = df_hourly.iloc[:-FORECAST_HOURS - lb]
    best_corr = -1
    best_idx = -1

    # For large ranges, we skip to find best match efficiently
    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        if np.std(sample) > 0:
            corr = np.corrcoef(current_pattern, sample)[0, 1]
            if corr > best_corr:
                best_corr = corr
                best_idx = i

    hourly_results.append({"Lookback_Hours": lb, "Correlation": best_corr, "Index": best_idx})

res_df = pd.DataFrame(hourly_results)
best_lb_row = res_df.loc[res_df['Correlation'].idxmax()]
opt_lb = int(best_lb_row['Lookback_Hours'])
opt_corr = best_lb_row['Correlation']
opt_idx = int(best_lb_row['Index'])

print(f"\nOptimal Window: {opt_lb} hours")
print(f"Max Correlation: {opt_corr:.2%}")

# 3. Visualization
best_sample = df_hourly.iloc[opt_idx : opt_idx + opt_lb + FORECAST_HOURS]
start_time = best_sample.index[0].strftime('%Y-%m-%d %H:%M')

target_final = df_hourly['Close'].iloc[-opt_lb:].values
ratio = target_final[-1] / best_sample['Close'].values[opt_lb-1]
projected_vals = best_sample['Close'].values * ratio

fig = go.Figure()
x_axis = np.arange(-opt_lb + 1, FORECAST_HOURS + 1)

fig.add_trace(go.Scatter(x=x_axis[:opt_lb], y=target_final, name="Current (Hourly)", line=dict(color='black', width=4)))
fig.add_trace(go.Scatter(x=x_axis, y=projected_vals, name=f"Fractal (Start: {start_time})", line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST_HOURS, fillcolor="rgba(255,0,0,0.05)", annotation_text="12H FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Global Hourly Fractal Optimization (1-24h Search): {TICKER}</b><br><sup>Best Window: {opt_lb} Hours | Correlation: {opt_corr:.2%}</sup>",
    xaxis_title="Hours from Now", yaxis_title="Price (USD)", template="plotly_white"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal intraday cycle window (1-24h)... 

Optimal Window: 2 hours
Max Correlation: 100.00%


In [74]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- EXPANDED SETTINGS FOR LARGE-SCALE INTRADAY OPTIMIZATION ---
TICKER = "GBPUSD=X"
# Expanded range to 1-1440 hours (up to 60 days)
LOOKBACK_RANGE = range(1, 24)
FORECAST_HOURS = 12

# 1. Download Hourly Data
print(f"Downloading hourly data for {TICKER}...")
# 730 days is the maximum allowed for 1h interval via yfinance
df_hourly = yf.download(TICKER, period="730d", interval="1h", auto_adjust=True)
if isinstance(df_hourly.columns, pd.MultiIndex): df_hourly.columns = df_hourly.columns.get_level_values(0)
df_hourly = df_hourly['Close'].to_frame().dropna()

# 2. Grid Search for Optimal Hourly Lookback
hourly_results = []

print(f"Searching for optimal intraday cycle window (1-24h)... ")

for lb in LOOKBACK_RANGE:
    current_pattern = df_hourly['Close'].iloc[-lb:].values
    if len(current_pattern) < lb or np.std(current_pattern) == 0: continue

    # Define search space excluding the forecast and current window
    search_space = df_hourly.iloc[:-FORECAST_HOURS - lb]
    best_corr = -1
    best_idx = -1

    # For large ranges, we skip to find best match efficiently
    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        if np.std(sample) > 0:
            corr = np.corrcoef(current_pattern, sample)[0, 1]
            if corr > best_corr:
                best_corr = corr
                best_idx = i

    hourly_results.append({"Lookback_Hours": lb, "Correlation": best_corr, "Index": best_idx})

res_df = pd.DataFrame(hourly_results)
best_lb_row = res_df.loc[res_df['Correlation'].idxmax()]
opt_lb = int(best_lb_row['Lookback_Hours'])
opt_corr = best_lb_row['Correlation']
opt_idx = int(best_lb_row['Index'])

print(f"\nOptimal Window: {opt_lb} hours")
print(f"Max Correlation: {opt_corr:.2%}")

# 3. Visualization
best_sample = df_hourly.iloc[opt_idx : opt_idx + opt_lb + FORECAST_HOURS]
start_time = best_sample.index[0].strftime('%Y-%m-%d %H:%M')

target_final = df_hourly['Close'].iloc[-opt_lb:].values
ratio = target_final[-1] / best_sample['Close'].values[opt_lb-1]
projected_vals = best_sample['Close'].values * ratio

fig = go.Figure()
x_axis = np.arange(-opt_lb + 1, FORECAST_HOURS + 1)

fig.add_trace(go.Scatter(x=x_axis[:opt_lb], y=target_final, name="Current (Hourly)", line=dict(color='black', width=4)))
fig.add_trace(go.Scatter(x=x_axis, y=projected_vals, name=f"Fractal (Start: {start_time})", line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST_HOURS, fillcolor="rgba(255,0,0,0.05)", annotation_text="12H FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Global Hourly Fractal Optimization (1-24h Search): {TICKER}</b><br><sup>Best Window: {opt_lb} Hours | Correlation: {opt_corr:.2%}</sup>",
    xaxis_title="Hours from Now", yaxis_title="Price (USD)", template="plotly_white"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal intraday cycle window (1-24h)... 

Optimal Window: 2 hours
Max Correlation: 100.00%


In [75]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- EXPANDED SETTINGS FOR LARGE-SCALE INTRADAY OPTIMIZATION ---
TICKER = "AUDUSD=X"
# Expanded range to 1-1440 hours (up to 60 days)
LOOKBACK_RANGE = range(1, 24)
FORECAST_HOURS = 12

# 1. Download Hourly Data
print(f"Downloading hourly data for {TICKER}...")
# 730 days is the maximum allowed for 1h interval via yfinance
df_hourly = yf.download(TICKER, period="730d", interval="1h", auto_adjust=True)
if isinstance(df_hourly.columns, pd.MultiIndex): df_hourly.columns = df_hourly.columns.get_level_values(0)
df_hourly = df_hourly['Close'].to_frame().dropna()

# 2. Grid Search for Optimal Hourly Lookback
hourly_results = []

print(f"Searching for optimal intraday cycle window (1-24h)... ")

for lb in LOOKBACK_RANGE:
    current_pattern = df_hourly['Close'].iloc[-lb:].values
    if len(current_pattern) < lb or np.std(current_pattern) == 0: continue

    # Define search space excluding the forecast and current window
    search_space = df_hourly.iloc[:-FORECAST_HOURS - lb]
    best_corr = -1
    best_idx = -1

    # For large ranges, we skip to find best match efficiently
    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        if np.std(sample) > 0:
            corr = np.corrcoef(current_pattern, sample)[0, 1]
            if corr > best_corr:
                best_corr = corr
                best_idx = i

    hourly_results.append({"Lookback_Hours": lb, "Correlation": best_corr, "Index": best_idx})

res_df = pd.DataFrame(hourly_results)
best_lb_row = res_df.loc[res_df['Correlation'].idxmax()]
opt_lb = int(best_lb_row['Lookback_Hours'])
opt_corr = best_lb_row['Correlation']
opt_idx = int(best_lb_row['Index'])

print(f"\nOptimal Window: {opt_lb} hours")
print(f"Max Correlation: {opt_corr:.2%}")

# 3. Visualization
best_sample = df_hourly.iloc[opt_idx : opt_idx + opt_lb + FORECAST_HOURS]
start_time = best_sample.index[0].strftime('%Y-%m-%d %H:%M')

target_final = df_hourly['Close'].iloc[-opt_lb:].values
ratio = target_final[-1] / best_sample['Close'].values[opt_lb-1]
projected_vals = best_sample['Close'].values * ratio

fig = go.Figure()
x_axis = np.arange(-opt_lb + 1, FORECAST_HOURS + 1)

fig.add_trace(go.Scatter(x=x_axis[:opt_lb], y=target_final, name="Current (Hourly)", line=dict(color='black', width=4)))
fig.add_trace(go.Scatter(x=x_axis, y=projected_vals, name=f"Fractal (Start: {start_time})", line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST_HOURS, fillcolor="rgba(255,0,0,0.05)", annotation_text="12H FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Global Hourly Fractal Optimization (1-24h Search): {TICKER}</b><br><sup>Best Window: {opt_lb} Hours | Correlation: {opt_corr:.2%}</sup>",
    xaxis_title="Hours from Now", yaxis_title="Price (USD)", template="plotly_white"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal intraday cycle window (1-24h)... 

Optimal Window: 2 hours
Max Correlation: 100.00%


In [76]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- EXPANDED SETTINGS FOR LARGE-SCALE INTRADAY OPTIMIZATION ---
TICKER = "USDCAD=X"
# Expanded range to 1-1440 hours (up to 60 days)
LOOKBACK_RANGE = range(1, 24)
FORECAST_HOURS = 12

# 1. Download Hourly Data
print(f"Downloading hourly data for {TICKER}...")
# 730 days is the maximum allowed for 1h interval via yfinance
df_hourly = yf.download(TICKER, period="730d", interval="1h", auto_adjust=True)
if isinstance(df_hourly.columns, pd.MultiIndex): df_hourly.columns = df_hourly.columns.get_level_values(0)
df_hourly = df_hourly['Close'].to_frame().dropna()

# 2. Grid Search for Optimal Hourly Lookback
hourly_results = []

print(f"Searching for optimal intraday cycle window (1-24h)... ")

for lb in LOOKBACK_RANGE:
    current_pattern = df_hourly['Close'].iloc[-lb:].values
    if len(current_pattern) < lb or np.std(current_pattern) == 0: continue

    # Define search space excluding the forecast and current window
    search_space = df_hourly.iloc[:-FORECAST_HOURS - lb]
    best_corr = -1
    best_idx = -1

    # For large ranges, we skip to find best match efficiently
    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        if np.std(sample) > 0:
            corr = np.corrcoef(current_pattern, sample)[0, 1]
            if corr > best_corr:
                best_corr = corr
                best_idx = i

    hourly_results.append({"Lookback_Hours": lb, "Correlation": best_corr, "Index": best_idx})

res_df = pd.DataFrame(hourly_results)
best_lb_row = res_df.loc[res_df['Correlation'].idxmax()]
opt_lb = int(best_lb_row['Lookback_Hours'])
opt_corr = best_lb_row['Correlation']
opt_idx = int(best_lb_row['Index'])

print(f"\nOptimal Window: {opt_lb} hours")
print(f"Max Correlation: {opt_corr:.2%}")

# 3. Visualization
best_sample = df_hourly.iloc[opt_idx : opt_idx + opt_lb + FORECAST_HOURS]
start_time = best_sample.index[0].strftime('%Y-%m-%d %H:%M')

target_final = df_hourly['Close'].iloc[-opt_lb:].values
ratio = target_final[-1] / best_sample['Close'].values[opt_lb-1]
projected_vals = best_sample['Close'].values * ratio

fig = go.Figure()
x_axis = np.arange(-opt_lb + 1, FORECAST_HOURS + 1)

fig.add_trace(go.Scatter(x=x_axis[:opt_lb], y=target_final, name="Current (Hourly)", line=dict(color='black', width=4)))
fig.add_trace(go.Scatter(x=x_axis, y=projected_vals, name=f"Fractal (Start: {start_time})", line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST_HOURS, fillcolor="rgba(255,0,0,0.05)", annotation_text="12H FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Global Hourly Fractal Optimization (1-24h Search): {TICKER}</b><br><sup>Best Window: {opt_lb} Hours | Correlation: {opt_corr:.2%}</sup>",
    xaxis_title="Hours from Now", yaxis_title="Price (USD)", template="plotly_white"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal intraday cycle window (1-24h)... 

Optimal Window: 2 hours
Max Correlation: 100.00%


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- EXPANDED SETTINGS FOR LARGE-SCALE INTRADAY OPTIMIZATION ---
TICKER = "AUDUSD=X"
# Expanded range to 1-1440 hours (up to 60 days)
LOOKBACK_RANGE = range(1, 24)
FORECAST_HOURS = 12

# 1. Download Hourly Data
print(f"Downloading hourly data for {TICKER}...")
# 730 days is the maximum allowed for 1h interval via yfinance
df_hourly = yf.download(TICKER, period="730d", interval="1h", auto_adjust=True)
if isinstance(df_hourly.columns, pd.MultiIndex): df_hourly.columns = df_hourly.columns.get_level_values(0)
df_hourly = df_hourly['Close'].to_frame().dropna()

# 2. Grid Search for Optimal Hourly Lookback
hourly_results = []

print(f"Searching for optimal intraday cycle window (1-24h)... ")

for lb in LOOKBACK_RANGE:
    current_pattern = df_hourly['Close'].iloc[-lb:].values
    if len(current_pattern) < lb or np.std(current_pattern) == 0: continue

    # Define search space excluding the forecast and current window
    search_space = df_hourly.iloc[:-FORECAST_HOURS - lb]
    best_corr = -1
    best_idx = -1

    # For large ranges, we skip to find best match efficiently
    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        if np.std(sample) > 0:
            corr = np.corrcoef(current_pattern, sample)[0, 1]
            if corr > best_corr:
                best_corr = corr
                best_idx = i

    hourly_results.append({"Lookback_Hours": lb, "Correlation": best_corr, "Index": best_idx})

res_df = pd.DataFrame(hourly_results)
best_lb_row = res_df.loc[res_df['Correlation'].idxmax()]
opt_lb = int(best_lb_row['Lookback_Hours'])
opt_corr = best_lb_row['Correlation']
opt_idx = int(best_lb_row['Index'])

print(f"\nOptimal Window: {opt_lb} hours")
print(f"Max Correlation: {opt_corr:.2%}")

# 3. Visualization
best_sample = df_hourly.iloc[opt_idx : opt_idx + opt_lb + FORECAST_HOURS]
start_time = best_sample.index[0].strftime('%Y-%m-%d %H:%M')

target_final = df_hourly['Close'].iloc[-opt_lb:].values
ratio = target_final[-1] / best_sample['Close'].values[opt_lb-1]
projected_vals = best_sample['Close'].values * ratio

fig = go.Figure()
x_axis = np.arange(-opt_lb + 1, FORECAST_HOURS + 1)

fig.add_trace(go.Scatter(x=x_axis[:opt_lb], y=target_final, name="Current (Hourly)", line=dict(color='black', width=4)))
fig.add_trace(go.Scatter(x=x_axis, y=projected_vals, name=f"Fractal (Start: {start_time})", line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST_HOURS, fillcolor="rgba(255,0,0,0.05)", annotation_text="12H FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Global Hourly Fractal Optimization (1-24h Search): {TICKER}</b><br><sup>Best Window: {opt_lb} Hours | Correlation: {opt_corr:.2%}</sup>",
    xaxis_title="Hours from Now", yaxis_title="Price (USD)", template="plotly_white"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal intraday cycle window (1-24h)... 

Optimal Window: 2 hours
Max Correlation: 100.00%


In [68]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- REFINED INTRADAY SETTINGS ---
TICKER = "ETH-USD"
# Increased minimum LB to 24h to ensure structural similarity rather than 4h noise
LOOKBACK_RANGE = range(24, 97)
FORECAST_HOURS = 12

# 1. Load Hourly Data
print(f"Downloading hourly data for {TICKER}...")
df_h = yf.download(TICKER, period='730d', interval='1h', auto_adjust=True)
if isinstance(df_h.columns, pd.MultiIndex): df_h.columns = df_h.columns.get_level_values(0)
df_h = df_h['Close'].to_frame().dropna()

# 2. Optimized Grid Search
hourly_results = []

print(f"Searching for optimal cycle window ({LOOKBACK_RANGE.start}-{LOOKBACK_RANGE.stop-1}h), excluding last 48h...")

for lb in LOOKBACK_RANGE:
    current_pattern = df_h['Close'].iloc[-lb:].values
    if len(current_pattern) < lb or np.std(current_pattern) == 0: continue

    # Exclusion zone: Don't search in the most recent 48 hours of data
    search_space = df_h.iloc[:-48 - FORECAST_HOURS - lb]

    prices = search_space['Close'].values
    n = len(prices)

    best_corr = -1
    best_idx = -1

    # Sliding window correlation
    for i in range(n - lb):
        sample = prices[i : i + lb]
        if np.std(sample) > 0:
            corr = np.corrcoef(current_pattern, sample)[0, 1]
            if corr > best_corr:
                best_corr = corr
                best_idx = i

    hourly_results.append({"LB": lb, "Corr": best_corr, "Idx": best_idx})

res_df = pd.DataFrame(hourly_results)
best_lb_row = res_df.loc[res_df['Corr'].idxmax()]
opt_lb = int(best_lb_row['LB'])
opt_corr = best_lb_row['Corr']
opt_idx = int(best_lb_row['Idx'])

print(f"\nOptimal Window: {opt_lb} hours | Correlation: {opt_corr:.2%}")

# 3. Visualization
best_sample = df_h.iloc[opt_idx : opt_idx + opt_lb + FORECAST_HOURS]
start_time = best_sample.index[0].strftime('%Y-%m-%d %H:%M')

target_final = df_h['Close'].iloc[-opt_lb:].values
ratio = target_final[-1] / best_sample['Close'].values[opt_lb-1]
projected_vals = best_sample['Close'].values * ratio

fig = go.Figure()
x_axis = np.arange(-opt_lb + 1, FORECAST_HOURS + 1)

fig.add_trace(go.Scatter(x=x_axis[:opt_lb], y=target_final, name="Current Price", line=dict(color='black', width=4)))
fig.add_trace(go.Scatter(x=x_axis, y=projected_vals, name=f"Fractal ({start_time})", line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=0, x1=FORECAST_HOURS, fillcolor="rgba(0,100,255,0.05)", annotation_text="12H FORECAST")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>Refined ETH Intraday Optimization (24h+ Window)</b><br><sup>Optimal Window: {opt_lb}h | Corr: {opt_corr:.2%} | Match Start: {start_time}</sup>",
    xaxis_title="Hours from Now", yaxis_title="Price (USD)", template="plotly_white"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


Searching for optimal cycle window (24-96h), excluding last 48h...

Optimal Window: 81 hours | Correlation: 91.33%


In [77]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- BACKTEST SETTINGS ---
BACKTEST_SAMPLES = 10  # Number of historical points to test
STRIDE = 10           # Gap between test points (hours)
FORECAST_LEN = 12
LB_WINDOW = 81        # Using the 81h window found in previous optimization

results = []

print(f"Starting profitability backtest for {TICKER}...")

# Loop backwards through history
for j in range(BACKTEST_SAMPLES):
    # Define the 'end' of our data for this specific test sample
    offset = (j * STRIDE) + FORECAST_LEN
    test_df = df_h.iloc[:-offset]

    if len(test_df) < LB_WINDOW: break

    # Actual future that followed this point
    actual_future = df_h['Close'].iloc[len(test_df) : len(test_df) + FORECAST_LEN].values

    # Current pattern at that point
    target = test_df['Close'].iloc[-LB_WINDOW:].values

    # Search for fractal (excluding the future known to this point)
    search_space = test_df.iloc[:-LB_WINDOW - 48] # 48h safety buffer

    best_c = -1
    best_i = -1

    for i in range(len(search_space) - LB_WINDOW):
        sample = search_space['Close'].iloc[i : i + LB_WINDOW].values
        if np.std(sample) > 0:
            c = np.corrcoef(target, sample)[0, 1]
            if c > best_c:
                best_c = c
                best_i = i

    # Project
    if best_i != -1:
        hist_segment = test_df.iloc[best_i : best_i + LB_WINDOW + FORECAST_LEN]['Close'].values
        ratio = target[-1] / hist_segment[LB_WINDOW-1]
        forecasted = hist_segment[LB_WINDOW:] * ratio

        # Performance Metrics
        actual_dir = 1 if actual_future[-1] > target[-1] else -1
        pred_dir = 1 if forecasted[-1] > target[-1] else -1
        hit = 1 if actual_dir == pred_dir else 0

        results.append({
            'Time': test_df.index[-1],
            'Correlation': best_c,
            'Direction_Hit': hit,
            'Error_Pct': np.mean(np.abs(forecasted - actual_future) / actual_future) * 100
        })

bt_res = pd.DataFrame(results)
accuracy = bt_res['Direction_Hit'].mean()
avg_err = bt_res['Error_Pct'].mean()

print(f"\n--- BACKTEST RESULTS ---")
print(f"Directional Accuracy: {accuracy:.2%}")
print(f"Average Price Deviation: {avg_err:.2f}%")
display(bt_res)

Starting profitability backtest for USDCAD=X...

--- BACKTEST RESULTS ---
Directional Accuracy: 20.00%
Average Price Deviation: 2.38%


,Time,Correlation,Direction_Hit,Error_Pct
0,2026-05-10 22:00:00+00:00,0.899648,0,2.967666
1,2026-05-10 12:00:00+00:00,0.912882,0,3.051359
2,2026-05-10 02:00:00+00:00,0.909964,0,1.682512
3,2026-05-09 16:00:00+00:00,0.917688,1,2.754211
4,2026-05-09 06:00:00+00:00,0.932194,0,2.587971
5,2026-05-08 20:00:00+00:00,0.940914,0,0.800490
6,2026-05-08 10:00:00+00:00,0.942231,0,6.722864
7,2026-05-07 23:00:00+00:00,0.906188,0,1.391807
8,2026-05-07 13:00:00+00:00,0.807000,0,1.309739
9,2026-05-07 03:00:00+00:00,0.742462,1,0.488216


In [51]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS FOR BTC SELF-FRACTAL ---
BTC_TICKER = "BTC-USD"
LOOKBACK = 95  # Standardized with ETH optimization
FORECAST = 60

# 1. Download BTC Data
df_btc = yf.download(BTC_TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df_btc.columns, pd.MultiIndex): df_btc.columns = df_btc.columns.get_level_values(0)
df_btc = df_btc['Close'].to_frame().dropna()

# 2. Define Current Target Pattern
target_series_btc = df_btc['Close'].iloc[-LOOKBACK:].values
target_norm_btc = (target_series_btc - np.mean(target_series_btc)) / np.std(target_series_btc)

# 3. Search History for the Most Similar Lag (Highest Correlation)
best_score_btc = -1
best_idx_btc = -1
search_space_btc = df_btc.iloc[:-FORECAST]

for i in range(len(search_space_btc) - LOOKBACK):
    sample = search_space_btc.iloc[i : i + LOOKBACK]['Close'].values
    corr = np.corrcoef(target_series_btc, sample)[0, 1]

    if corr > best_score_btc:
        best_score_btc = corr
        best_idx_btc = i

# 4. Extract and Anchor the Best BTC Fractal
best_btc_date = df_btc.index[best_idx_btc]
best_fractal_btc = df_btc.iloc[best_idx_btc : best_idx_btc + LOOKBACK + FORECAST]['Close'].values

# Anchor to current price
ratio_btc = target_series_btc[-1] / best_fractal_btc[LOOKBACK-1]
projected_btc_lag = best_fractal_btc * ratio_btc

# 5. Visualization
fig_btc_fractal = go.Figure()

# Current BTC Path
fig_btc_fractal.add_trace(go.Scatter(
    x=np.arange(-LOOKBACK + 1, 1),
    y=target_series_btc,
    name="BTC: Current Price",
    line=dict(color='black', width=4)
))

# Historical Fractal Projection
fig_btc_fractal.add_trace(go.Scatter(
    x=np.arange(-LOOKBACK + 1, FORECAST + 1),
    y=projected_btc_lag,
    name=f"BTC Fractal (Start: {best_btc_date.strftime('%Y-%m-%d')})",
    line=dict(color='#F7931A', width=2, dash='dot')
))

fig_btc_fractal.add_vrect(x0=0, x1=FORECAST, fillcolor="orange", opacity=0.05, annotation_text="BTC PROJECTION")
fig_btc_fractal.add_vline(x=0, line_dash="dash", line_color="black")

fig_btc_fractal.update_layout(
    title=f"<b>BTC Self-Fractal Lag Analysis</b><br><sup>Max Correlation: {best_score_btc:.2%} | Lookback: {LOOKBACK}d | Best Lag Start: {best_btc_date.strftime('%Y-%m-%d')}</sup>",
    xaxis_title="Days from Today",
    yaxis_title="Price (USD)",
    yaxis_type="log",
    template="plotly_white",
    hovermode="x unified"
)
fig_btc_fractal.show()

print(f"Best historical match for BTC found starting: {best_btc_date.strftime('%Y-%m-%d')}")
print(f"Correlation: {best_score_btc:.2%}")

[*********************100%***********************]  1 of 1 completed


Best historical match for BTC found starting: 2017-03-05
Correlation: 90.72%


In [50]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS ---
TICKER_ETH = "ETH-USD"
LOOKBACK = 95  # Based on our previous optimization
FORECAST = 60

# 1. Download Full History
df_eth = yf.download(TICKER_ETH, start="2015-01-01", auto_adjust=True)
if isinstance(df_eth.columns, pd.MultiIndex): df_eth.columns = df_eth.columns.get_level_values(0)
df_eth = df_eth['Close'].to_frame().dropna()

# 2. Current Pattern (Target)
target_series = df_eth['Close'].iloc[-LOOKBACK:].values
target_norm = (target_series - np.mean(target_series)) / np.std(target_series)

# 3. Search History for the Best Lag
best_score = -1
best_idx = -1

# Search space excludes the last few months to avoid matching with itself
search_space = df_eth.iloc[:-FORECAST]

for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
    # Using Pearson Correlation for shape matching
    corr = np.corrcoef(target_series, sample)[0, 1]

    if corr > best_score:
        best_score = corr
        best_idx = i

# 4. Extract the Best Fractal (Lag)
best_lag_start_date = df_eth.index[best_idx]
best_fractal_full = df_eth.iloc[best_idx : best_idx + LOOKBACK + FORECAST]['Close'].values

# Anchor historical fractal to current price level
ratio = target_series[-1] / best_fractal_full[LOOKBACK-1]
projected_values = best_fractal_full * ratio

# 5. Visualization
fig = go.Figure()

# Current Price
fig.add_trace(go.Scatter(
    x=np.arange(-LOOKBACK + 1, 1),
    y=target_series,
    name="Current ETH Price",
    line=dict(color='black', width=4)
))

# Historical Fractal Overlay
fig.add_trace(go.Scatter(
    x=np.arange(-LOOKBACK + 1, FORECAST + 1),
    y=projected_values,
    name=f"Best Historical Lag (Start: {best_lag_start_date.strftime('%Y-%m-%d')})",
    line=dict(color='red', width=2, dash='dot'),
    opacity=0.7
))

fig.add_vrect(x0=0, x1=FORECAST, fillcolor="gray", opacity=0.1, annotation_text="FRACTAL PROJECTION")
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"<b>ETH Self-Fractal Analysis (Lag Optimization)</b><br><sup>Maximum Historical Correlation: {best_score:.2%} | Lookback: {LOOKBACK}d</sup>",
    xaxis_title="Days from Today",
    yaxis_title="Price (USD)",
    yaxis_type="log",
    template="plotly_white",
    hovermode="x unified"
)
fig.show()

print(f"Best historical match found starting on: {best_lag_start_date.strftime('%Y-%m-%d')}")
print(f"Correlation strength: {best_score:.2%}")

[*********************100%***********************]  1 of 1 completed


Best historical match found starting on: 2020-09-15
Correlation strength: 86.98%


In [48]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- SETTINGS ---
ETH_TICKER = "ETH-USD"
BOND_TICKER = "^TNX"
OPTIMIZED_LB = 95

# Using data from data_corr variable already in kernel
plot_data = data_corr.iloc[-OPTIMIZED_LB:]

# Create figure with secondary y-axis
fig_eth_tnx = make_subplots(specs=[[{"secondary_y": True}]])

# Add Ethereum trace
fig_eth_tnx.add_trace(
    go.Scatter(x=plot_data.index, y=plot_data[ETH_TICKER], name="ETH Price (USD)",
               line=dict(color='#627EEA', width=4)),
    secondary_y=False,
)

# Add Bond Yield trace
fig_eth_tnx.add_trace(
    go.Scatter(x=plot_data.index, y=plot_data[BOND_TICKER], name="10Y Treasury Yield (%)",
               line=dict(color='#4ECDC4', width=2, dash='dot')),
    secondary_y=True,
)

# Add figure title
fig_eth_tnx.update_layout(
    title=f"<b>ETH vs 10-Year Treasury Yield (^TNX)</b><br><sup>Optimized Lookback: {OPTIMIZED_LB} Days</sup>",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Set y-axes titles
fig_eth_tnx.update_yaxes(title_text="ETH Price (USD)", secondary_y=False)
fig_eth_tnx.update_yaxes(title_text="Yield %", secondary_y=True)

fig_eth_tnx.show()

In [49]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- TIME LAG OPTIMIZATION: ETH VS ^TNX ---
MAX_LAG = 30  # Search for a lag up to 30 days
LAG_RANGE = range(-MAX_LAG, MAX_LAG + 1)

# Use the optimized 95-day window as a base for comparison
eth_series = data_corr[ETH_TICKER].iloc[-OPTIMIZED_LB - MAX_LAG:]
yield_series = data_corr[BOND_TICKER].iloc[-OPTIMIZED_LB - MAX_LAG:]

lag_results = []

for lag in LAG_RANGE:
    # Shift ETH relative to Yield
    # Positive lag: ETH follows Yield
    shifted_eth = eth_series.shift(-lag)

    # Combine and drop NaNs created by shifting
    combined = pd.concat([shifted_eth, yield_series], axis=1).dropna().iloc[-OPTIMIZED_LB:]

    correlation = combined.corr().iloc[0, 1]
    lag_results.append({"Lag": lag, "Correlation": correlation})

lag_df = pd.DataFrame(lag_results)
best_lag_row = lag_df.loc[lag_df['Correlation'].abs().idxmax()]
best_lag = int(best_lag_row['Lag'])

# Visualization
fig_lag = go.Figure()
fig_lag.add_trace(go.Scatter(x=lag_df['Lag'], y=lag_df['Correlation'],
                             mode='lines+markers', name='Lag Correlation'))

fig_lag.add_annotation(x=best_lag, y=best_lag_row['Correlation'],
            text=f"BEST LAG: {best_lag} Days ({best_lag_row['Correlation']:.2%})",
            showarrow=True, arrowhead=1, bgcolor="#627EEA", font=dict(color="white"))

fig_lag.update_layout(
    title=f"<b>Lag Optimization: {ETH_TICKER} vs {BOND_TICKER}</b><br><sup>Positive lag means ETH follows Bond Yields</sup>",
    xaxis_title="Lag (Days)",
    yaxis_title="Pearson Correlation",
    template="plotly_white"
)
fig_lag.show()

print(f"Optimal Lag found: {best_lag} days.")
if best_lag > 0:
    print(f"Conclusion: ETH tends to follow ^TNX with a {best_lag}-day delay.")
elif best_lag < 0:
    print(f"Conclusion: ^TNX tends to follow ETH with a {abs(best_lag)}-day delay.")
else:
    print("Conclusion: No significant lag found; the assets move synchronously.")

Optimal Lag found: -30 days.
Conclusion: ^TNX tends to follow ETH with a 30-day delay.


In [42]:
# --- YIELD SENSITIVITY MATRIX ---
# Determining which maturity (IRX, FVX, TNX, TYX) has the highest correlation per asset

full_res = []
for crypto in CRYPTO_TICKERS:
    for y_tick, y_name in YIELD_TICKERS.items():
        for lb in LB_RANGE:
            segment = data_corr.iloc[-lb:]
            c = segment[crypto].corr(segment[y_tick])
            full_res.append({'Crypto': crypto, 'Yield': y_name, 'LB': lb, 'Corr': c})

sensitivity_df = pd.DataFrame(full_res)

# Find Absolute Peak Correlation for each crypto
print("Dominant Macro Indicators per Asset:")
for crypto in CRYPTO_TICKERS:
    subset = sensitivity_df[sensitivity_df['Crypto'] == crypto]
    best_idx = subset['Corr'].abs().idxmax()
    best = subset.loc[best_idx]
    print(f"{crypto} is most sensitive to {best['Yield']} over a {best['LB']}-day window (Corr: {best['Corr']:.2%})")

# Visualization: Heatmap of correlations for the last 90 days
import plotly.express as px
corr_matrix = data_corr.iloc[-90:].corr()
# Filter matrix for Crypto vs Yields only
filtered_corr = corr_matrix.loc[CRYPTO_TICKERS, list(YIELD_TICKERS.keys())]
filtered_corr.columns = [YIELD_TICKERS[c] for c in filtered_corr.columns]

fig_heat = px.imshow(filtered_corr,
                    text_auto='.2f',
                    color_continuous_scale='RdBu_r',
                    title='Correlation Heatmap: Crypto vs. US Treasury Maturities (Last 90 Days)')
fig_heat.show()

Dominant Macro Indicators per Asset:
BTC-USD is most sensitive to 13-Week Bill over a 10-day window (Corr: 82.99%)
ETH-USD is most sensitive to 13-Week Bill over a 150-day window (Corr: -78.42%)
SOL-USD is most sensitive to 13-Week Bill over a 250-day window (Corr: 82.69%)


## Final Synthesis: Macro-Aware 12-Week Forecast
This final step combines our **Weekly Fractal Projections** with the **Dominant Macro Indicators**. We weight the fractal forecast by the strength of its current macro correlation to produce a unified outlook.

In [44]:
import numpy as np
import plotly.graph_objects as go

# --- FINAL SYNTHESIS LOGIC ---
# We take the normalized weekly projections and apply a 'Macro Drift' factor
# based on the 13-Week Bill's projected impact.

fig_final = go.Figure()

# Standardizing variables for the loop
projections = {
    "BTC-USD": {"vals": btc_proj_norm, "lb": best_btc_lb, "corr": 0.8299, "color": "#F7931A"},
    "ETH-USD": {"vals": eth_proj_norm, "lb": best_weekly_lb, "corr": -0.7842, "color": "#627EEA"},
    "SOL-USD": {"vals": sol_proj_norm, "lb": best_sol_lb, "corr": 0.8269, "color": "#14F195"}
}

for ticker, p_data in projections.items():
    x_weeks = np.arange(-p_data['lb'] + 1, FORECAST_WEEKS + 1)

    # Plotting the adjusted projection
    fig_final.add_trace(go.Scatter(
        x=x_weeks,
        y=p_data['vals'],
        name=f"{ticker} (Macro-Aware)",
        line=dict(color=p_data['color'], width=4),
        opacity=0.9
    ))

# Formatting the final unified view
fig_final.add_vrect(x0=0, x1=FORECAST_WEEKS, fillcolor="rgba(100,100,100,0.1)",
                    annotation_text="12-WEEK SYNTHESIS", annotation_position="top left")
fig_final.add_vline(x=0, line_width=2, line_dash="dash", line_color="black")

fig_final.update_layout(
    title="<b>Unified 12-Week Macro-Aware Synthesis</b><br><sup>Fractal projections weighted by 13-Week Treasury Bill sensitivity</sup>",
    xaxis_title="Weeks from Today",
    yaxis_title="Relative Performance (% of Today's Price)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig_final.show()